In [1]:
# ============================================================
# TRACE THE ACE — EVIDENCE PACK BUILDER
# CELL 0 — ENVIRONMENT / PATHS / CONFIG BOOTSTRAP
# ============================================================

from pathlib import Path
import gc
import hashlib
import json
import platform
import sys
from datetime import datetime, timezone

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


print("=" * 80)
print("TRACE THE ACE — EVIDENCE PACK BUILDER")
print("CELL 0 — ENVIRONMENT / PATHS / CONFIG BOOTSTRAP")
print("=" * 80)


# ============================================================
# 0. PROJECT ROOT
# ============================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

assert PROJECT_ROOT.exists(), (
    f"PROJECT_ROOT does not exist: {PROJECT_ROOT}"
)

assert PROJECT_ROOT.is_dir(), (
    f"PROJECT_ROOT is not a directory: {PROJECT_ROOT}"
)


# ============================================================
# 1. CANONICAL DATA ROOT
# ============================================================

CANONICAL_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "01_data_foundation"
    / "03_integrity"
    / "canonical"
)

assert CANONICAL_ROOT.exists(), (
    f"Canonical root does not exist: {CANONICAL_ROOT}"
)

assert CANONICAL_ROOT.is_dir(), (
    f"Canonical root is not a directory: {CANONICAL_ROOT}"
)


# ============================================================
# 2. FROZEN CROSS-ENCODER ROOT
# ============================================================

CROSS_ENCODER_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "02_retrieval"
    / "cross_encoder"
)

CROSS_ENCODER_FROZEN_ROOT = (
    CROSS_ENCODER_ROOT
    / "frozen"
)

assert CROSS_ENCODER_FROZEN_ROOT.exists(), (
    "Frozen Cross-Encoder artifact root does not exist:\n"
    f"{CROSS_ENCODER_FROZEN_ROOT}"
)


# ============================================================
# 3. CROSS-ENCODER FROZEN ARTIFACTS
# ============================================================

CROSS_ENCODER_RANKED_PATH = (
    CROSS_ENCODER_FROZEN_ROOT
    / "cross_encoder_ranked_candidates.parquet"
)

CROSS_ENCODER_MANIFEST_PATH = (
    CROSS_ENCODER_FROZEN_ROOT
    / "cell6_freeze_manifest.json"
)

assert CROSS_ENCODER_RANKED_PATH.exists(), (
    "Frozen Cross-Encoder ranked candidate artifact missing:\n"
    f"{CROSS_ENCODER_RANKED_PATH}"
)

assert CROSS_ENCODER_MANIFEST_PATH.exists(), (
    "Frozen Cross-Encoder manifest missing:\n"
    f"{CROSS_ENCODER_MANIFEST_PATH}"
)


# ============================================================
# 4. CANONICAL ARTIFACTS
# ============================================================

RESPONSES_PATH = (
    CANONICAL_ROOT
    / "responses.parquet"
)

TURNS_PATH = (
    CANONICAL_ROOT
    / "turns.parquet"
)

SESSIONS_PATH = (
    CANONICAL_ROOT
    / "sessions.parquet"
)

OBJECTIVES_PATH = (
    CANONICAL_ROOT
    / "objectives.parquet"
)

for artifact_path in [
    RESPONSES_PATH,
    TURNS_PATH,
    SESSIONS_PATH,
    OBJECTIVES_PATH,
]:
    assert artifact_path.exists(), (
        f"Canonical artifact missing:\n{artifact_path}"
    )


# ============================================================
# 5. EVIDENCE PACK OUTPUT ROOT
# ============================================================

EVIDENCE_PACK_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "03_evidence_pack"
)

EVIDENCE_PACK_FROZEN_ROOT = (
    EVIDENCE_PACK_ROOT
    / "frozen"
)

EVIDENCE_PACK_AUDIT_ROOT = (
    EVIDENCE_PACK_ROOT
    / "audit"
)

EVIDENCE_PACK_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

EVIDENCE_PACK_FROZEN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

EVIDENCE_PACK_AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 6. OUTPUT ARTIFACT PATHS
# ============================================================

EVIDENCE_PACK_PATH = (
    EVIDENCE_PACK_FROZEN_ROOT
    / "evidence_packs.parquet"
)

EVIDENCE_PACK_MANIFEST_PATH = (
    EVIDENCE_PACK_FROZEN_ROOT
    / "evidence_pack_manifest.json"
)

EVIDENCE_PACK_AUDIT_PATH = (
    EVIDENCE_PACK_AUDIT_ROOT
    / "evidence_pack_audit.parquet"
)


# ============================================================
# 7. ARCHITECTURE CONFIG
# ============================================================

EVIDENCE_PACK_MAX_TOKENS = 2048

EVIDENCE_PACK_VERSION = "1.0"

# Number of reranked turns considered before evidence selection.
# This is a selection budget, NOT the final token budget.
RERANKED_TURN_BUDGET = 32

# Role-aware selection targets.
STUDENT_TOP_K = 8
TUTOR_TOP_K = 4

# Temporal neighbours around selected evidence.
NEIGHBOUR_RADIUS = 1

# Keep final student evidence explicitly represented.
INCLUDE_FINAL_STUDENT = True

# Keep objective text explicitly represented.
INCLUDE_OBJECTIVE = True


# ============================================================
# 8. REQUIRED CROSS-ENCODER SCHEMA
# ============================================================

REQUIRED_CE_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "cross_encoder_score",
    "cross_encoder_rank",
]

# We do NOT load the 2.48M-row artifact into memory yet.
# Schema validation is performed through Parquet metadata.
CE_PARQUET = pq.ParquetFile(
    CROSS_ENCODER_RANKED_PATH
)

CE_SCHEMA_COLUMNS = (
    CE_PARQUET.schema_arrow.names
)

missing_ce_columns = sorted(
    set(REQUIRED_CE_COLUMNS)
    - set(CE_SCHEMA_COLUMNS)
)

assert not missing_ce_columns, (
    "Frozen Cross-Encoder artifact is missing required columns:\n"
    f"{missing_ce_columns}"
)


# ============================================================
# 9. FROZEN CROSS-ENCODER MANIFEST
# ============================================================

with open(
    CROSS_ENCODER_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as f:
    CROSS_ENCODER_MANIFEST = json.load(f)

assert isinstance(
    CROSS_ENCODER_MANIFEST,
    dict,
), "Cross-Encoder manifest must be a JSON object."


# Accept only an explicitly frozen/complete artifact.
manifest_text = json.dumps(
    CROSS_ENCODER_MANIFEST
).upper()

assert (
    "FROZEN" in manifest_text
    or
    CROSS_ENCODER_MANIFEST.get("status") == "FROZEN"
), (
    "Cross-Encoder manifest does not indicate a frozen artifact."
)


# ============================================================
# 10. ENVIRONMENT SNAPSHOT
# ============================================================

EVIDENCE_PACK_ENV = {
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "python_version": sys.version,

    "platform": platform.platform(),

    "pandas_version": pd.__version__,

    "pyarrow_version": pa.__version__,

    "project_root": str(PROJECT_ROOT),

    "canonical_root": str(CANONICAL_ROOT),

    "cross_encoder_ranked_path": str(
        CROSS_ENCODER_RANKED_PATH
    ),

    "cross_encoder_manifest_path": str(
        CROSS_ENCODER_MANIFEST_PATH
    ),

    "evidence_pack_root": str(
        EVIDENCE_PACK_ROOT
    ),

    "evidence_pack_version":
        EVIDENCE_PACK_VERSION,

    "max_tokens":
        EVIDENCE_PACK_MAX_TOKENS,

    "reranked_turn_budget":
        RERANKED_TURN_BUDGET,

    "student_top_k":
        STUDENT_TOP_K,

    "tutor_top_k":
        TUTOR_TOP_K,

    "neighbour_radius":
        NEIGHBOUR_RADIUS,
}


# ============================================================
# 11. BASIC SOURCE POPULATION CHECK
# ============================================================

responses_pf = pq.ParquetFile(
    RESPONSES_PATH
)

turns_pf = pq.ParquetFile(
    TURNS_PATH
)

sessions_pf = pq.ParquetFile(
    SESSIONS_PATH
)

objectives_pf = pq.ParquetFile(
    OBJECTIVES_PATH
)

CE_ROWS = int(
    CE_PARQUET.metadata.num_rows
)

RESPONSE_ROWS = int(
    responses_pf.metadata.num_rows
)

TURN_ROWS = int(
    turns_pf.metadata.num_rows
)

SESSION_ROWS = int(
    sessions_pf.metadata.num_rows
)

OBJECTIVE_ROWS = int(
    objectives_pf.metadata.num_rows
)


# ============================================================
# 12. HARD EXPECTATIONS
# ============================================================

assert RESPONSE_ROWS == 35_072, (
    f"Unexpected response population: {RESPONSE_ROWS}"
)

assert TURN_ROWS == 6_139_854, (
    f"Unexpected turn population: {TURN_ROWS}"
)

assert SESSION_ROWS == 22_821, (
    f"Unexpected session population: {SESSION_ROWS}"
)

assert OBJECTIVE_ROWS == 398, (
    f"Unexpected objective population: {OBJECTIVE_ROWS}"
)

assert CE_ROWS > 0, (
    "Frozen Cross-Encoder artifact contains zero rows."
)


# ============================================================
# 13. IMMUTABILITY RULE
# ============================================================

# Cell 0 must NEVER overwrite canonical data or the frozen
# Cross-Encoder artifact.

CANONICAL_OUTPUTS_FORBIDDEN = {
    RESPONSES_PATH,
    TURNS_PATH,
    SESSIONS_PATH,
    OBJECTIVES_PATH,
    CROSS_ENCODER_RANKED_PATH,
    CROSS_ENCODER_MANIFEST_PATH,
}


# ============================================================
# 14. STATUS
# ============================================================

EVIDENCE_PACK_BOOTSTRAP_READY = True


print("\n" + "=" * 80)
print("CELL 0 — BOOTSTRAP SUMMARY")
print("=" * 80)

print(
    f"Project root             : {PROJECT_ROOT}"
)

print(
    f"Canonical responses      : {RESPONSE_ROWS:,}"
)

print(
    f"Canonical turns          : {TURN_ROWS:,}"
)

print(
    f"Canonical sessions       : {SESSION_ROWS:,}"
)

print(
    f"Canonical objectives     : {OBJECTIVE_ROWS:,}"
)

print(
    f"Cross-Encoder rows       : {CE_ROWS:,}"
)

print(
    f"Evidence max tokens      : {EVIDENCE_PACK_MAX_TOKENS}"
)

print(
    f"Reranked turn budget     : {RERANKED_TURN_BUDGET}"
)

print(
    f"Student top-k            : {STUDENT_TOP_K}"
)

print(
    f"Tutor top-k              : {TUTOR_TOP_K}"
)

print(
    f"Neighbour radius         : {NEIGHBOUR_RADIUS}"
)

print(
    f"CE schema validated      : {not missing_ce_columns}"
)

print(
    f"Bootstrap ready          : {EVIDENCE_PACK_BOOTSTRAP_READY}"
)

print("=" * 80)


# Cleanup metadata-only handles.
del responses_pf
del turns_pf
del sessions_pf
del objectives_pf
del CE_PARQUET

gc.collect()

TRACE THE ACE — EVIDENCE PACK BUILDER
CELL 0 — ENVIRONMENT / PATHS / CONFIG BOOTSTRAP

CELL 0 — BOOTSTRAP SUMMARY
Project root             : D:\Competition\Trace-the-race-local
Canonical responses      : 35,072
Canonical turns          : 6,139,854
Canonical sessions       : 22,821
Canonical objectives     : 398
Cross-Encoder rows       : 2,482,137
Evidence max tokens      : 2048
Reranked turn budget     : 32
Student top-k            : 8
Tutor top-k              : 4
Neighbour radius         : 1
CE schema validated      : True
Bootstrap ready          : True


0

In [2]:
# ==============================================================================
# TRACE THE ACE — EVIDENCE PACK BUILDER
# CELL 1 — FROZEN CROSS-ENCODER CONTRACT + INPUT INTEGRITY AUDIT
# ==============================================================================

import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


print("=" * 80)
print(
    "TRACE THE ACE — EVIDENCE PACK BUILDER"
)
print(
    "CELL 1 — FROZEN CROSS-ENCODER CONTRACT + INPUT INTEGRITY AUDIT"
)
print("=" * 80)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    "EVIDENCE_PACK_BOOTSTRAP_READY" in globals()
    and EVIDENCE_PACK_BOOTSTRAP_READY is True
), (
    "Cell 0 dependency failed."
)

print("\n" + "=" * 80)
print("CELL 0 DEPENDENCY")
print("=" * 80)

print(
    "Cell 0 dependency : PASS"
)


# ==============================================================================
# 2. REOPEN FROZEN CROSS-ENCODER ARTIFACT
# ==============================================================================

assert CROSS_ENCODER_RANKED_PATH.exists(), (
    "Frozen Cross-Encoder parquet is missing:\n"
    f"{CROSS_ENCODER_RANKED_PATH}"
)

assert CROSS_ENCODER_MANIFEST_PATH.exists(), (
    "Frozen Cross-Encoder manifest is missing:\n"
    f"{CROSS_ENCODER_MANIFEST_PATH}"
)

ce_pf = pq.ParquetFile(
    CROSS_ENCODER_RANKED_PATH
)

ce_schema = ce_pf.schema_arrow


print("\n" + "=" * 80)
print("FROZEN CROSS-ENCODER INPUT")
print("=" * 80)

print(
    "Candidate parquet :",
    CROSS_ENCODER_RANKED_PATH,
)

print(
    "Freeze manifest    :",
    CROSS_ENCODER_MANIFEST_PATH,
)

print(
    "Artifact rows      :",
    f"{ce_pf.metadata.num_rows:,}",
)

print(
    "Artifact columns   :",
    ce_schema.names,
)


# ==============================================================================
# 3. EXACT POPULATION CONTRACT
# ==============================================================================

EXPECTED_CE_ROWS = 2_482_137

assert (
    ce_pf.metadata.num_rows
    ==
    EXPECTED_CE_ROWS
), (
    "Unexpected frozen Cross-Encoder population.\n"
    f"Observed: {ce_pf.metadata.num_rows:,}\n"
    f"Expected: {EXPECTED_CE_ROWS:,}"
)

print("\n" + "=" * 80)
print("POPULATION CONTRACT")
print("=" * 80)

print(
    "Observed rows :",
    f"{ce_pf.metadata.num_rows:,}",
)

print(
    "Expected rows :",
    f"{EXPECTED_CE_ROWS:,}",
)

print(
    "Population contract : PASS"
)


# ==============================================================================
# 4. EXACT COLUMN CONTRACT
# ==============================================================================

EXPECTED_CE_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "cross_encoder_score",
    "cross_encoder_rank",
]

assert (
    ce_schema.names
    ==
    EXPECTED_CE_COLUMNS
), (
    "Frozen Cross-Encoder column contract mismatch.\n"
    f"Observed: {ce_schema.names}\n"
    f"Expected: {EXPECTED_CE_COLUMNS}"
)

print("\n" + "=" * 80)
print("COLUMN CONTRACT")
print("=" * 80)

print(
    "Observed columns:",
    ce_schema.names,
)

print(
    "Column contract : PASS"
)


# ==============================================================================
# 5. ARROW TYPE CONTRACT
# ==============================================================================

EXPECTED_ARROW_TYPES = {
    "response_id": "string",
    "session_id": "string",
    "objective_uid": "string",
    "fold": "int64",
    "turn_uid": "string",
    "role": "string",
    "turn_index": "int64",
    "cross_encoder_score": "double",
    "cross_encoder_rank": "int64",
}

observed_arrow_types = {
    field.name: str(field.type)
    for field in ce_schema
}

for column, expected_type in EXPECTED_ARROW_TYPES.items():

    observed_type = observed_arrow_types[column]

    assert (
        observed_type
        ==
        expected_type
    ), (
        f"Unexpected Arrow type for {column}.\n"
        f"Observed: {observed_type}\n"
        f"Expected: {expected_type}"
    )

print("\n" + "=" * 80)
print("ARROW TYPE CONTRACT")
print("=" * 80)

for column in EXPECTED_CE_COLUMNS:
    print(
        f"{column:24s}:",
        observed_arrow_types[column],
    )

print(
    "Arrow type contract : PASS"
)


# ==============================================================================
# 6. FROZEN MANIFEST CONTRACT
# ==============================================================================

with open(
    CROSS_ENCODER_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as f:
    ce_manifest = json.load(f)

assert isinstance(
    ce_manifest,
    dict,
), (
    "Cross-Encoder manifest must be a JSON object."
)

manifest_status = ce_manifest.get(
    "status"
)

assert (
    manifest_status == "FROZEN"
), (
    "Cross-Encoder manifest is not FROZEN.\n"
    f"Observed status: {manifest_status}"
)

print("\n" + "=" * 80)
print("FROZEN MANIFEST CONTRACT")
print("=" * 80)

print(
    "JSON valid : PASS"
)

print(
    "Status     :",
    manifest_status,
)

print(
    "Manifest contract : PASS"
)


# ==============================================================================
# 7. READ FIRST BATCH ONLY
# ==============================================================================

# Do NOT load the complete 2.48M-row artifact into pandas.
# This cell intentionally performs contract checks using a bounded sample.

sample_table = ce_pf.read_row_group(
    0
)

sample_df = (
    sample_table
    .to_pandas()
)

assert (
    len(sample_df) > 0
), (
    "First Cross-Encoder row group is empty."
)

print("\n" + "=" * 80)
print("BOUNDED SAMPLE")
print("=" * 80)

print(
    "First row-group rows :",
    f"{len(sample_df):,}",
)

print(
    "Bounded sample load : PASS"
)


# ==============================================================================
# 8. SCORE FINITENESS
# ==============================================================================

sample_scores = pd.to_numeric(
    sample_df[
        "cross_encoder_score"
    ],
    errors="coerce",
)

assert (
    sample_scores.notna().all()
), (
    "Non-numeric / NaN Cross-Encoder scores "
    "found in bounded sample."
)

assert (
    np.isfinite(
        sample_scores.to_numpy(
            dtype=np.float64
        )
    ).all()
), (
    "Non-finite Cross-Encoder scores "
    "found in bounded sample."
)

print("\n" + "=" * 80)
print("SCORE CONTRACT")
print("=" * 80)

print(
    "Sample score finite : PASS"
)

print(
    "Sample score min    :",
    float(sample_scores.min()),
)

print(
    "Sample score max    :",
    float(sample_scores.max()),
)


# ==============================================================================
# 9. FOLD CONTRACT
# ==============================================================================

sample_folds = pd.to_numeric(
    sample_df[
        "fold"
    ],
    errors="coerce",
)

assert (
    sample_folds.notna().all()
), (
    "NaN fold values found."
)

assert (
    set(
        sample_folds.astype(int).unique()
    )
    <=
    {0, 1, 2, 3, 4}
), (
    "Unexpected fold value found in "
    "Cross-Encoder sample."
)

print("\n" + "=" * 80)
print("FOLD CONTRACT")
print("=" * 80)

print(
    "Observed sample folds :",
    sorted(
        sample_folds.astype(int).unique()
        .tolist()
    ),
)

print(
    "Fold contract : PASS"
)


# ==============================================================================
# 10. RANK CONTRACT
# ==============================================================================

sample_ranks = pd.to_numeric(
    sample_df[
        "cross_encoder_rank"
    ],
    errors="coerce",
)

assert (
    sample_ranks.notna().all()
), (
    "NaN Cross-Encoder ranks found."
)

assert (
    (
        sample_ranks
        >=
        1
    )
    &
    (
        sample_ranks
        <=
        100
    )
).all(), (
    "Cross-Encoder rank outside expected "
    "positive bounded range in sample."
)

print("\n" + "=" * 80)
print("RANK CONTRACT")
print("=" * 80)

print(
    "Rank min :",
    int(sample_ranks.min()),
)

print(
    "Rank max :",
    int(sample_ranks.max()),
)

print(
    "Rank contract : PASS"
)


# ==============================================================================
# 11. IDENTITY NULL CONTRACT
# ==============================================================================

IDENTITY_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "turn_uid",
    "role",
]

for column in IDENTITY_COLUMNS:

    assert (
        sample_df[column].notna().all()
    ), (
        f"Null identity value found in {column}."
    )

print("\n" + "=" * 80)
print("IDENTITY NULL CONTRACT")
print("=" * 80)

print(
    "Identity columns :",
    IDENTITY_COLUMNS,
)

print(
    "Null identity values : PASS"
)


# ==============================================================================
# 12. RESPONSE / OBJECTIVE KEY CONTRACT
# ==============================================================================

sample_key = (
    sample_df[
        [
            "response_id",
            "objective_uid",
        ]
    ]
    .astype(str)
)

assert (
    sample_key.notna().all().all()
), (
    "Invalid response/objective identity "
    "in bounded sample."
)

print("\n" + "=" * 80)
print("RESPONSE / OBJECTIVE KEY CONTRACT")
print("=" * 80)

print(
    "Response-objective identity : PASS"
)


# ==============================================================================
# 13. READ-ONLY CONTRACT
# ==============================================================================

assert (
    CROSS_ENCODER_RANKED_PATH
    in
    CANONICAL_OUTPUTS_FORBIDDEN
), (
    "Frozen Cross-Encoder artifact is not protected "
    "by the read-only contract."
)

assert (
    RESPONSES_PATH
    in
    CANONICAL_OUTPUTS_FORBIDDEN
)

assert (
    TURNS_PATH
    in
    CANONICAL_OUTPUTS_FORBIDDEN
)

assert (
    SESSIONS_PATH
    in
    CANONICAL_OUTPUTS_FORBIDDEN
)

assert (
    OBJECTIVES_PATH
    in
    CANONICAL_OUTPUTS_FORBIDDEN
)

print("\n" + "=" * 80)
print("READ-ONLY CONTRACT")
print("=" * 80)

print(
    "Frozen upstream artifacts protected : PASS"
)

print(
    "Canonical artifacts protected       : PASS"
)


# ==============================================================================
# 14. FINAL CELL 1 GATE
# ==============================================================================

EVIDENCE_PACK_CELL_1_READY = True

print("\n" + "=" * 80)
print(
    "EVIDENCE PACK CELL 1 — "
    "FROZEN CROSS-ENCODER INPUT AUDIT: PASS"
)
print("=" * 80)


# ==============================================================================
# 15. MEMORY CLEANUP
# ==============================================================================

del sample_table
del sample_df
del sample_scores
del sample_folds
del sample_ranks
del sample_key
del ce_pf
del ce_schema
del observed_arrow_types
del ce_manifest

gc.collect()

print(
    "Cell 1 memory cleanup: PASS"
)

TRACE THE ACE — EVIDENCE PACK BUILDER
CELL 1 — FROZEN CROSS-ENCODER CONTRACT + INPUT INTEGRITY AUDIT

CELL 0 DEPENDENCY
Cell 0 dependency : PASS

FROZEN CROSS-ENCODER INPUT
Candidate parquet : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\frozen\cross_encoder_ranked_candidates.parquet
Freeze manifest    : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\frozen\cell6_freeze_manifest.json
Artifact rows      : 2,482,137
Artifact columns   : ['response_id', 'session_id', 'objective_uid', 'fold', 'turn_uid', 'role', 'turn_index', 'cross_encoder_score', 'cross_encoder_rank']

POPULATION CONTRACT
Observed rows : 2,482,137
Expected rows : 2,482,137
Population contract : PASS

COLUMN CONTRACT
Observed columns: ['response_id', 'session_id', 'objective_uid', 'fold', 'turn_uid', 'role', 'turn_index', 'cross_encoder_score', 'cross_encoder_rank']
Column contract : PASS

ARROW TYPE CONTRACT
response_id             : string

In [3]:
# ==============================================================================
# TRACE THE ACE — EVIDENCE PACK BUILDER
# CELL 2 — RERANKED EVIDENCE DISTRIBUTION + ROLE / TEMPORAL COVERAGE AUDIT
# ==============================================================================

import gc
from collections import defaultdict

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


print("=" * 80)
print(
    "TRACE THE ACE — EVIDENCE PACK BUILDER"
)
print(
    "CELL 2 — RERANKED EVIDENCE DISTRIBUTION + ROLE / TEMPORAL COVERAGE AUDIT"
)
print("=" * 80)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    "EVIDENCE_PACK_CELL_1_READY" in globals()
    and EVIDENCE_PACK_CELL_1_READY is True
), (
    "Cell 1 dependency failed."
)

print("\n" + "=" * 80)
print("DEPENDENCY GATE")
print("=" * 80)

print(
    "Cell 1 dependency : PASS"
)


# ==============================================================================
# 2. OPEN FROZEN CROSS-ENCODER ARTIFACT
# ==============================================================================

assert CROSS_ENCODER_RANKED_PATH.exists(), (
    "Frozen Cross-Encoder parquet is missing."
)

ce_pf = pq.ParquetFile(
    CROSS_ENCODER_RANKED_PATH
)

total_rows = int(
    ce_pf.metadata.num_rows
)

print("\n" + "=" * 80)
print("FROZEN INPUT")
print("=" * 80)

print(
    "Rows:",
    f"{total_rows:,}",
)

print(
    "Row groups:",
    f"{ce_pf.num_row_groups:,}",
)


# ==============================================================================
# 3. AUDIT COLUMNS ONLY
# ==============================================================================

AUDIT_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "cross_encoder_score",
    "cross_encoder_rank",
]

assert (
    set(AUDIT_COLUMNS)
    <=
    set(ce_pf.schema_arrow.names)
), (
    "Required audit columns are missing."
)


# ==============================================================================
# 4. MEMORY-SAFE BATCH SCAN
# ==============================================================================

# We intentionally process RecordBatches instead of converting the complete
# 2.48M-row artifact into one pandas DataFrame.

BATCH_SIZE = 100_000

group_stats = {}

role_counts = defaultdict(
    lambda: defaultdict(int)
)

rank_score_rows = []

global_rank_min = None
global_rank_max = None
global_score_min = None
global_score_max = None

null_counts = defaultdict(int)

observed_rows = 0


def update_min(current, value):
    if current is None:
        return value
    return min(current, value)


def update_max(current, value):
    if current is None:
        return value
    return max(current, value)


# ==============================================================================
# 5. STREAM THROUGH FROZEN ARTIFACT
# ==============================================================================

scanner = ce_pf.iter_batches(
    batch_size=BATCH_SIZE,
    columns=AUDIT_COLUMNS,
)

batch_number = 0

for batch in scanner:

    batch_number += 1

    df = batch.to_pandas()

    observed_rows += len(df)

    # --------------------------------------------------------------------------
    # Basic null accounting
    # --------------------------------------------------------------------------

    for column in AUDIT_COLUMNS:

        null_counts[column] += int(
            df[column].isna().sum()
        )

    # --------------------------------------------------------------------------
    # Numeric conversion
    # --------------------------------------------------------------------------

    rank_values = pd.to_numeric(
        df["cross_encoder_rank"],
        errors="coerce",
    )

    score_values = pd.to_numeric(
        df["cross_encoder_score"],
        errors="coerce",
    )

    assert rank_values.notna().all(), (
        "Non-numeric / NaN rank detected."
    )

    assert score_values.notna().all(), (
        "Non-numeric / NaN score detected."
    )

    rank_array = rank_values.to_numpy(
        dtype=np.float64
    )

    score_array = score_values.to_numpy(
        dtype=np.float64
    )

    assert np.isfinite(
        rank_array
    ).all(), (
        "Non-finite rank detected."
    )

    assert np.isfinite(
        score_array
    ).all(), (
        "Non-finite Cross-Encoder score detected."
    )

    batch_rank_min = float(
        rank_array.min()
    )

    batch_rank_max = float(
        rank_array.max()
    )

    batch_score_min = float(
        score_array.min()
    )

    batch_score_max = float(
        score_array.max()
    )

    global_rank_min = update_min(
        global_rank_min,
        batch_rank_min,
    )

    global_rank_max = update_max(
        global_rank_max,
        batch_rank_max,
    )

    global_score_min = update_min(
        global_score_min,
        batch_score_min,
    )

    global_score_max = update_max(
        global_score_max,
        batch_score_max,
    )

    # --------------------------------------------------------------------------
    # Role distribution
    # --------------------------------------------------------------------------

    role_batch = (
        df[
            [
                "response_id",
                "objective_uid",
                "role",
            ]
        ]
        .groupby(
            [
                "response_id",
                "objective_uid",
                "role",
            ],
            dropna=False,
        )
        .size()
    )

    for (
        response_id,
        objective_uid,
        role,
    ), count in role_batch.items():

        key = (
            str(response_id),
            str(objective_uid),
        )

        role_counts[key][
            str(role)
        ] += int(count)

    # --------------------------------------------------------------------------
    # Per response-objective rank/score summary
    # --------------------------------------------------------------------------

    batch_summary = (
        df
        .groupby(
            [
                "response_id",
                "objective_uid",
            ],
            dropna=False,
        )
        .agg(
            candidate_count=(
                "turn_uid",
                "size",
            ),
            rank_min=(
                "cross_encoder_rank",
                "min",
            ),
            rank_max=(
                "cross_encoder_rank",
                "max",
            ),
            score_max=(
                "cross_encoder_score",
                "max",
            ),
            score_min=(
                "cross_encoder_score",
                "min",
            ),
            turn_index_min=(
                "turn_index",
                "min",
            ),
            turn_index_max=(
                "turn_index",
                "max",
            ),
        )
        .reset_index()
    )

    rank_score_rows.append(
        batch_summary
    )

    del df
    del role_batch
    del batch_summary

    if batch_number % 10 == 0:
        print(
            f"Processed batches: {batch_number:,} | "
            f"Rows: {observed_rows:,}"
        )

    gc.collect()


# ==============================================================================
# 6. EXACT POPULATION CHECK
# ==============================================================================

assert (
    observed_rows
    ==
    total_rows
), (
    "Streaming row count does not match parquet metadata."
)

assert (
    observed_rows
    ==
    2_482_137
), (
    "Unexpected Cross-Encoder row population."
)


print("\n" + "=" * 80)
print("POPULATION AUDIT")
print("=" * 80)

print(
    "Observed streamed rows:",
    f"{observed_rows:,}",
)

print(
    "Parquet metadata rows :",
    f"{total_rows:,}",
)

print(
    "Population audit : PASS"
)


# ==============================================================================
# 7. NULL AUDIT
# ==============================================================================

print("\n" + "=" * 80)
print("NULL AUDIT")
print("=" * 80)

for column in AUDIT_COLUMNS:

    count = int(
        null_counts[column]
    )

    print(
        f"{column:24s}:",
        f"{count:,}",
    )

    assert (
        count == 0
    ), (
        f"Null values detected in {column}."
    )

print(
    "Null audit : PASS"
)


# ==============================================================================
# 8. GLOBAL SCORE / RANK DISTRIBUTION
# ==============================================================================

assert global_rank_min is not None
assert global_rank_max is not None
assert global_score_min is not None
assert global_score_max is not None

print("\n" + "=" * 80)
print("GLOBAL RANK / SCORE DISTRIBUTION")
print("=" * 80)

print(
    "Rank min:",
    global_rank_min,
)

print(
    "Rank max:",
    global_rank_max,
)

print(
    "Score min:",
    global_score_min,
)

print(
    "Score max:",
    global_score_max,
)

print(
    "Rank / score finiteness : PASS"
)


# ==============================================================================
# 9. RESPONSE-OBJECTIVE SUMMARY
# ==============================================================================

summary_df = pd.concat(
    rank_score_rows,
    ignore_index=True,
)

del rank_score_rows

response_objective_summary = (
    summary_df
    .groupby(
        [
            "response_id",
            "objective_uid",
        ],
        as_index=False,
    )
    .agg(
        candidate_count=(
            "candidate_count",
            "sum",
        ),
        rank_min=(
            "rank_min",
            "min",
        ),
        rank_max=(
            "rank_max",
            "max",
        ),
        score_max=(
            "score_max",
            "max",
        ),
        score_min=(
            "score_min",
            "min",
        ),
        turn_index_min=(
            "turn_index_min",
            "min",
        ),
        turn_index_max=(
            "turn_index_max",
            "max",
        ),
    )
)

del summary_df

response_objective_summary["response_id"] = (
    response_objective_summary[
        "response_id"
    ].astype(str)
)

response_objective_summary["objective_uid"] = (
    response_objective_summary[
        "objective_uid"
    ].astype(str)
)


# ==============================================================================
# 10. GROUP POPULATION CONTRACT
# ==============================================================================

group_count = len(
    response_objective_summary
)

print("\n" + "=" * 80)
print("RESPONSE-OBJECTIVE GROUP CONTRACT")
print("=" * 80)

print(
    "Unique response-objective groups:",
    f"{group_count:,}",
)

assert (
    group_count
    ==
    35_072
), (
    "Unexpected number of response-objective groups."
)

print(
    "Expected groups:",
    "35,072",
)

print(
    "Group population : PASS"
)


# ==============================================================================
# 11. CANDIDATE COUNT DISTRIBUTION
# ==============================================================================

candidate_min = int(
    response_objective_summary[
        "candidate_count"
    ].min()
)

candidate_max = int(
    response_objective_summary[
        "candidate_count"
    ].max()
)

candidate_mean = float(
    response_objective_summary[
        "candidate_count"
    ].mean()
)

candidate_median = float(
    response_objective_summary[
        "candidate_count"
    ].median()
)

print("\n" + "=" * 80)
print("CANDIDATE COUNT DISTRIBUTION")
print("=" * 80)

print(
    "Min    :",
    candidate_min,
)

print(
    "Max    :",
    candidate_max,
)

print(
    "Mean   :",
    candidate_mean,
)

print(
    "Median :",
    candidate_median,
)

assert (
    candidate_min > 0
), (
    "At least one response-objective group "
    "has zero candidates."
)

print(
    "Non-empty candidate coverage : PASS"
)


# ==============================================================================
# 12. ROLE DISTRIBUTION
# ==============================================================================

role_distribution = defaultdict(int)

for key, counts in role_counts.items():

    for role, count in counts.items():

        role_distribution[
            role
        ] += int(count)

print("\n" + "=" * 80)
print("GLOBAL ROLE DISTRIBUTION")
print("=" * 80)

for role in sorted(
    role_distribution.keys()
):

    print(
        f"{role:20s}:",
        f"{role_distribution[role]:,}"
    )

print(
    "Role distribution audit : PASS"
)


# ==============================================================================
# 13. ROLE COVERAGE PER RESPONSE-OBJECTIVE
# ==============================================================================

group_role_rows = []

for (
    response_id,
    objective_uid,
), counts in role_counts.items():

    row = {
        "response_id": response_id,
        "objective_uid": objective_uid,
    }

    for role, count in counts.items():

        safe_role = (
            str(role)
            .strip()
            .lower()
            .replace(
                " ",
                "_",
            )
        )

        row[
            f"role_{safe_role}_count"
        ] = int(count)

    group_role_rows.append(
        row
    )

role_coverage_df = pd.DataFrame(
    group_role_rows
)

del group_role_rows
del role_counts

if not role_coverage_df.empty:

    role_columns = [
        column
        for column in role_coverage_df.columns
        if column.endswith("_count")
    ]

    role_coverage_df[
        role_columns
    ] = (
        role_coverage_df[
            role_columns
        ]
        .fillna(0)
        .astype(int)
    )

else:

    role_columns = []


print("\n" + "=" * 80)
print("ROLE COVERAGE")
print("=" * 80)

print(
    "Role count columns:",
    role_columns,
)

print(
    "Role coverage table rows:",
    f"{len(role_coverage_df):,}",
)

assert (
    len(role_coverage_df)
    ==
    35_072
), (
    "Role coverage does not contain "
    "all response-objective groups."
)

print(
    "Role coverage : PASS"
)


# ==============================================================================
# 14. TOP-RANK ROLE AUDIT
# ==============================================================================

# Read only the first few rank positions needed for diagnostic analysis.
# We do not yet decide the final evidence-selection policy.

TOP_RANK_AUDIT = 10

top_rank_rows = []

scanner = ce_pf.iter_batches(
    batch_size=BATCH_SIZE,
    columns=[
        "response_id",
        "objective_uid",
        "turn_uid",
        "role",
        "turn_index",
        "cross_encoder_score",
        "cross_encoder_rank",
    ],
)

for batch in scanner:

    df = batch.to_pandas()

    rank_values = pd.to_numeric(
        df["cross_encoder_rank"],
        errors="coerce",
    )

    top_df = df[
        rank_values
        <=
        TOP_RANK_AUDIT
    ].copy()

    if not top_df.empty:
        top_rank_rows.append(
            top_df
        )

    del df
    del top_df

    gc.collect()


top_rank_df = pd.concat(
    top_rank_rows,
    ignore_index=True,
)

del top_rank_rows

print("\n" + "=" * 80)
print("TOP-RANK ROLE AUDIT")
print("=" * 80)

print(
    "Rank threshold:",
    TOP_RANK_AUDIT,
)

print(
    "Rows observed within threshold:",
    f"{len(top_rank_df):,}",
)

top_role_distribution = (
    top_rank_df[
        "role"
    ]
    .value_counts(
        dropna=False
    )
)

for role, count in (
    top_role_distribution
    .items()
):

    print(
        f"{str(role):20s}:",
        f"{int(count):,}"
    )

print(
    "Top-rank role audit : PASS"
)


# ==============================================================================
# 15. TEMPORAL COVERAGE AUDIT
# ==============================================================================

temporal_span_min = int(
    response_objective_summary[
        "turn_index_min"
    ].min()
)

temporal_span_max = int(
    response_objective_summary[
        "turn_index_max"
    ].max()
)

print("\n" + "=" * 80)
print("TEMPORAL COVERAGE")
print("=" * 80)

print(
    "Global minimum turn index:",
    temporal_span_min,
)

print(
    "Global maximum turn index:",
    temporal_span_max,
)

assert (
    temporal_span_min
    >=
    0
), (
    "Negative turn index detected."
)

print(
    "Temporal index validity : PASS"
)


# ==============================================================================
# 16. SAVE AUDIT ARTIFACTS
# ==============================================================================

AUDIT_GROUP_PATH = (
    EVIDENCE_PACK_AUDIT_ROOT
    / "reranked_group_distribution.parquet"
)

AUDIT_ROLE_PATH = (
    EVIDENCE_PACK_AUDIT_ROOT
    / "reranked_role_coverage.parquet"
)

AUDIT_TOP_ROLE_PATH = (
    EVIDENCE_PACK_AUDIT_ROOT
    / "top_rank_role_distribution.parquet"
)

response_objective_summary.to_parquet(
    AUDIT_GROUP_PATH,
    index=False,
)

role_coverage_df.to_parquet(
    AUDIT_ROLE_PATH,
    index=False,
)

(
    top_role_distribution
    .rename(
        "row_count"
    )
    .reset_index()
    .rename(
        columns={
            "index": "role"
        }
    )
    .to_parquet(
        AUDIT_TOP_ROLE_PATH,
        index=False,
    )
)


# ==============================================================================
# 17. CELL 2 STATUS
# ==============================================================================

EVIDENCE_PACK_CELL_2_READY = True

print("\n" + "=" * 80)
print(
    "EVIDENCE PACK CELL 2 — "
    "RERANKED DISTRIBUTION / ROLE / TEMPORAL AUDIT: PASS"
)
print("=" * 80)

print(
    "Group audit:",
    AUDIT_GROUP_PATH,
)

print(
    "Role audit:",
    AUDIT_ROLE_PATH,
)

print(
    "Top-rank role audit:",
    AUDIT_TOP_ROLE_PATH,
)


# ==============================================================================
# 18. MEMORY CLEANUP
# ==============================================================================

del ce_pf
del response_objective_summary
del role_coverage_df
del top_rank_df
del top_role_distribution
del scanner

gc.collect()

print(
    "Cell 2 memory cleanup: PASS"
)

TRACE THE ACE — EVIDENCE PACK BUILDER
CELL 2 — RERANKED EVIDENCE DISTRIBUTION + ROLE / TEMPORAL COVERAGE AUDIT

DEPENDENCY GATE
Cell 1 dependency : PASS

FROZEN INPUT
Rows: 2,482,137
Row groups: 50
Processed batches: 10 | Rows: 1,000,000
Processed batches: 20 | Rows: 2,000,000

POPULATION AUDIT
Observed streamed rows: 2,482,137
Parquet metadata rows : 2,482,137
Population audit : PASS

NULL AUDIT
response_id             : 0
session_id              : 0
objective_uid           : 0
fold                    : 0
turn_uid                : 0
role                    : 0
turn_index              : 0
cross_encoder_score     : 0
cross_encoder_rank      : 0
Null audit : PASS

GLOBAL RANK / SCORE DISTRIBUTION
Rank min: 1.0
Rank max: 96.0
Score min: -11.526229858398438
Score max: 10.52168083190918
Rank / score finiteness : PASS

RESPONSE-OBJECTIVE GROUP CONTRACT
Unique response-objective groups: 35,072
Expected groups: 35,072
Group population : PASS

CANDIDATE COUNT DISTRIBUTION
Min    : 15
Max    : 9

In [13]:
# ==============================================================================
# TRACE THE ACE — EVIDENCE PACK BUILDER
# CELL 3 — EVIDENCE SELECTION POLICY + SELECTION CONTRACT AUDIT
# ==============================================================================

import gc
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds


print("=" * 80)
print("TRACE THE ACE — EVIDENCE PACK BUILDER")
print("CELL 3 — EVIDENCE SELECTION POLICY + SELECTION CONTRACT AUDIT")
print("=" * 80)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    "EVIDENCE_PACK_CELL_2_READY" in globals()
    and EVIDENCE_PACK_CELL_2_READY is True
), (
    "Cell 2 dependency failed."
)

print("\n" + "=" * 80)
print("DEPENDENCY GATE")
print("=" * 80)

print(
    "Cell 2 dependency : PASS"
)


# ==============================================================================
# 2. LOAD FROZEN CROSS-ENCODER ARTIFACT
# ==============================================================================

assert CROSS_ENCODER_RANKED_PATH.exists(), (
    "Frozen Cross-Encoder parquet is missing."
)

ce_pf = pq.ParquetFile(
    CROSS_ENCODER_RANKED_PATH
)

TOTAL_CE_ROWS = int(
    ce_pf.metadata.num_rows
)

print("\n" + "=" * 80)
print("FROZEN INPUT")
print("=" * 80)

print(
    "Cross-Encoder rows:",
    f"{TOTAL_CE_ROWS:,}",
)


# ==============================================================================
# 3. SELECTION POLICY
# ==============================================================================

# IMPORTANT:
# These are explicit selection rules for constructing an evidence candidate
# set. They are NOT yet the final token-budget packing rules.

SELECTION_RANK_WINDOW = 16

STUDENT_SELECTION_K = 8
TUTOR_SELECTION_K = 4

NEIGHBOUR_RADIUS = 1

FINAL_STUDENT_WINDOW = 5

MAX_SELECTED_TURNS = 24


# Canonical role aliases.
# We do not assume every role string has identical formatting.
STUDENT_ROLE_ALIASES = {
    "student",
    "learner",
    "user",
}

TUTOR_ROLE_ALIASES = {
    "tutor",
    "teacher",
    "assistant",
}


def canonical_role(value):
    if pd.isna(value):
        return "unknown"

    value = (
        str(value)
        .strip()
        .lower()
    )

    if value in STUDENT_ROLE_ALIASES:
        return "student"

    if value in TUTOR_ROLE_ALIASES:
        return "tutor"

    return value


print("\n" + "=" * 80)
print("SELECTION POLICY")
print("=" * 80)

print(
    "Rank window:",
    SELECTION_RANK_WINDOW,
)

print(
    "Student top-k:",
    STUDENT_SELECTION_K,
)

print(
    "Tutor top-k:",
    TUTOR_SELECTION_K,
)

print(
    "Neighbour radius:",
    NEIGHBOUR_RADIUS,
)

print(
    "Final student window:",
    FINAL_STUDENT_WINDOW,
)

print(
    "Maximum selected turns:",
    MAX_SELECTED_TURNS,
)


# ==============================================================================
# 4. READ RANKED CANDIDATES IN A MEMORY-SAFE STREAM
# ==============================================================================

READ_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "cross_encoder_score",
    "cross_encoder_rank",
]


BATCH_SIZE = 100_000


# Store only candidates inside the diagnostic rank window.
# We intentionally do NOT retain all 2.48M rows.

selected_window_batches = []

scanner = ce_pf.iter_batches(
    batch_size=BATCH_SIZE,
    columns=READ_COLUMNS,
)

processed_rows = 0

for batch in scanner:

    df = batch.to_pandas()

    processed_rows += len(df)

    ranks = pd.to_numeric(
        df["cross_encoder_rank"],
        errors="coerce",
    )

    assert ranks.notna().all(), (
        "Invalid rank encountered."
    )

    window_df = df[
        ranks
        <=
        SELECTION_RANK_WINDOW
    ].copy()

    if not window_df.empty:
        selected_window_batches.append(
            window_df
        )

    del df
    del window_df

    gc.collect()


assert (
    processed_rows
    ==
    TOTAL_CE_ROWS
), (
    "Streaming population mismatch."
)

candidate_window_df = pd.concat(
    selected_window_batches,
    ignore_index=True,
)

del selected_window_batches
del scanner

gc.collect()


print("\n" + "=" * 80)
print("RANK WINDOW")
print("=" * 80)

print(
    "Processed rows:",
    f"{processed_rows:,}",
)

print(
    "Window rows:",
    f"{len(candidate_window_df):,}",
)

assert (
    len(candidate_window_df)
    >
    0
), (
    "Rank-window candidate set is empty."
)

print(
    "Rank-window extraction : PASS"
)


# ==============================================================================
# 5. ROLE NORMALIZATION
# ==============================================================================

candidate_window_df[
    "canonical_role"
] = (
    candidate_window_df[
        "role"
    ]
    .map(canonical_role)
)


print("\n" + "=" * 80)
print("ROLE NORMALIZATION")
print("=" * 80)

role_counts = (
    candidate_window_df[
        "canonical_role"
    ]
    .value_counts(
        dropna=False
    )
)

for role, count in role_counts.items():

    print(
        f"{str(role):20s}:",
        f"{int(count):,}"
    )

print(
    "Role normalization : PASS"
)


# ==============================================================================
# 6. HARD IDENTITY CONTRACT
# ==============================================================================

IDENTITY_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "turn_uid",
]

for column in IDENTITY_COLUMNS:

    assert (
        candidate_window_df[
            column
        ]
        .notna()
        .all()
    ), (
        f"Null value in identity column: {column}"
    )


assert (
    candidate_window_df[
        "turn_index"
    ]
    .notna()
    .all()
), (
    "Null turn_index encountered."
)

assert (
    candidate_window_df[
        "cross_encoder_score"
    ]
    .notna()
    .all()
), (
    "Null Cross-Encoder score encountered."
)

print("\n" + "=" * 80)
print("IDENTITY CONTRACT")
print("=" * 80)

print(
    "Identity columns : PASS"
)

print(
    "Turn index        : PASS"
)

print(
    "Cross-Encoder score: PASS"
)


# ==============================================================================
# 7. RESPONSE-OBJECTIVE GROUPING
# ==============================================================================

GROUP_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
]


grouped_candidates = (
    candidate_window_df
    .sort_values(
        [
            "response_id",
            "objective_uid",
            "cross_encoder_rank",
            "turn_index",
            "turn_uid",
        ],
        kind="mergesort",
    )
    .groupby(
        GROUP_COLUMNS,
        sort=False,
        dropna=False,
    )
)


# ==============================================================================
# 8. DETERMINISTIC SELECTION
# ==============================================================================

selection_rows = []

selection_diagnostics = []


for (
    response_id,
    session_id,
    objective_uid,
), group in grouped_candidates:

    group = group.copy()

    # --------------------------------------------------------------------------
    # Student candidates
    # --------------------------------------------------------------------------

    student_candidates = (
        group[
            group[
                "canonical_role"
            ]
            ==
            "student"
        ]
        .sort_values(
            [
                "cross_encoder_rank",
                "turn_index",
                "turn_uid",
            ],
            kind="mergesort",
        )
    )

    selected_student = (
        student_candidates
        .head(
            STUDENT_SELECTION_K
        )
    )

    # --------------------------------------------------------------------------
    # Tutor candidates
    # --------------------------------------------------------------------------

    tutor_candidates = (
        group[
            group[
                "canonical_role"
            ]
            ==
            "tutor"
        ]
        .sort_values(
            [
                "cross_encoder_rank",
                "turn_index",
                "turn_uid",
            ],
            kind="mergesort",
        )
    )

    selected_tutor = (
        tutor_candidates
        .head(
            TUTOR_SELECTION_K
        )
    )

    # --------------------------------------------------------------------------
    # Initial selected set
    # --------------------------------------------------------------------------

    selected_parts = []

    if not selected_student.empty:
        selected_parts.append(
            selected_student
        )

    if not selected_tutor.empty:
        selected_parts.append(
            selected_tutor
        )

    if selected_parts:

        selected = pd.concat(
            selected_parts,
            ignore_index=False,
        )

    else:

        selected = group.head(
            0
        ).copy()

    # --------------------------------------------------------------------------
    # Deterministic deduplication
    # --------------------------------------------------------------------------

    selected = (
        selected
        .sort_values(
            [
                "cross_encoder_rank",
                "turn_index",
                "turn_uid",
            ],
            kind="mergesort",
        )
        .drop_duplicates(
            subset=[
                "response_id",
                "session_id",
                "objective_uid",
                "turn_uid",
            ],
            keep="first",
        )
    )

    # --------------------------------------------------------------------------
    # Hard selected-turn cap
    # --------------------------------------------------------------------------

    selected = selected.head(
        MAX_SELECTED_TURNS
    )

    # --------------------------------------------------------------------------
    # Diagnostics
    # --------------------------------------------------------------------------

    student_count = int(
        (
            selected[
                "canonical_role"
            ]
            ==
            "student"
        )
        .sum()
    )

    tutor_count = int(
        (
            selected[
                "canonical_role"
            ]
            ==
            "tutor"
        )
        .sum()
    )

    unknown_count = int(
        (
            selected[
                "canonical_role"
            ]
            == "unknown"
        )
        .sum()
    )

    selection_diagnostics.append(
        {
            "response_id": str(
                response_id
            ),
            "session_id": str(
                session_id
            ),
            "objective_uid": str(
                objective_uid
            ),
            "rank_window_candidates": int(
                len(group)
            ),
            "student_available": int(
                len(student_candidates)
            ),
            "tutor_available": int(
                len(tutor_candidates)
            ),
            "student_selected": student_count,
            "tutor_selected": tutor_count,
            "unknown_selected": unknown_count,
            "selected_total": int(
                len(selected)
            ),
        }
    )

    # --------------------------------------------------------------------------
    # Selection rows
    # --------------------------------------------------------------------------

    if not selected.empty:

        selected = selected.copy()

        selected[
            "selection_source"
        ] = "cross_encoder_rank_window"

        selection_rows.append(
            selected
        )


# ==============================================================================
# 9. BUILD DIAGNOSTIC TABLES
# ==============================================================================

selection_diagnostics_df = pd.DataFrame(
    selection_diagnostics
)

del selection_diagnostics

if selection_rows:

    selected_candidate_df = pd.concat(
        selection_rows,
        ignore_index=True,
    )

else:

    selected_candidate_df = (
        candidate_window_df
        .head(0)
        .copy()
    )

del selection_rows
del grouped_candidates

gc.collect()


# ==============================================================================
# 10. EXACT GROUP COVERAGE
# ==============================================================================

EXPECTED_GROUPS = 35_072

observed_groups = int(
    selection_diagnostics_df[
        [
            "response_id",
            "objective_uid",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print("\n" + "=" * 80)
print("SELECTION GROUP COVERAGE")
print("=" * 80)

print(
    "Expected response-objective groups:",
    f"{EXPECTED_GROUPS:,}",
)

print(
    "Observed groups:",
    f"{observed_groups:,}",
)

assert (
    observed_groups
    ==
    EXPECTED_GROUPS
), (
    "Selection diagnostics do not cover "
    "all response-objective groups."
)

print(
    "Group coverage : PASS"
)


# ==============================================================================
# 11. EMPTY-SELECTION AUDIT
# ==============================================================================

empty_selection_mask = (
    selection_diagnostics_df[
        "selected_total"
    ]
    ==
    0
)

empty_selection_count = int(
    empty_selection_mask.sum()
)

print("\n" + "=" * 80)
print("EMPTY SELECTION AUDIT")
print("=" * 80)

print(
    "Empty selections:",
    f"{empty_selection_count:,}",
)

assert (
    empty_selection_count
    ==
    0
), (
    "At least one response-objective group "
    "has no selected evidence."
)

print(
    "Empty selection audit : PASS"
)


# ==============================================================================
# 12. STUDENT EVIDENCE COVERAGE
# ==============================================================================

no_student_mask = (
    selection_diagnostics_df[
        "student_selected"
    ]
    ==
    0
)

no_student_count = int(
    no_student_mask.sum()
)

student_selected_total = int(
    selection_diagnostics_df[
        "student_selected"
    ]
    .sum()
)

print("\n" + "=" * 80)
print("STUDENT EVIDENCE COVERAGE")
print("=" * 80)

print(
    "Groups without selected student evidence:",
    f"{no_student_count:,}",
)

print(
    "Total selected student turns:",
    f"{student_selected_total:,}",
)

# This is intentionally diagnostic rather than a hard failure.
# Some objectives may have only tutor/context evidence in the ranked window.
print(
    "Student coverage diagnostic : RECORDED"
)


# ==============================================================================
# 13. TUTOR EVIDENCE COVERAGE
# ==============================================================================

no_tutor_mask = (
    selection_diagnostics_df[
        "tutor_selected"
    ]
    ==
    0
)

no_tutor_count = int(
    no_tutor_mask.sum()
)

tutor_selected_total = int(
    selection_diagnostics_df[
        "tutor_selected"
    ]
    .sum()
)

print("\n" + "=" * 80)
print("TUTOR / CONTEXT COVERAGE")
print("=" * 80)

print(
    "Groups without selected tutor evidence:",
    f"{no_tutor_count:,}",
)

print(
    "Total selected tutor turns:",
    f"{tutor_selected_total:,}",
)

print(
    "Tutor coverage diagnostic : RECORDED"
)


# ==============================================================================
# 14. SELECTION SIZE CONTRACT
# ==============================================================================

selected_min = int(
    selection_diagnostics_df[
        "selected_total"
    ].min()
)

selected_max = int(
    selection_diagnostics_df[
        "selected_total"
    ].max()
)

selected_mean = float(
    selection_diagnostics_df[
        "selected_total"
    ].mean()
)

selected_median = float(
    selection_diagnostics_df[
        "selected_total"
    ].median()
)

print("\n" + "=" * 80)
print("SELECTION SIZE DISTRIBUTION")
print("=" * 80)

print(
    "Min    :",
    selected_min,
)

print(
    "Max    :",
    selected_max,
)

print(
    "Mean   :",
    selected_mean,
)

print(
    "Median :",
    selected_median,
)

assert (
    selected_min
    >=
    1
), (
    "Invalid zero-sized selection."
)

assert (
    selected_max
    <=
    MAX_SELECTED_TURNS
), (
    "Selected-turn cap violated."
)

print(
    "Selection size contract : PASS"
)


# ==============================================================================
# 15. DUPLICATE IDENTITY AUDIT
# ==============================================================================

selected_identity = [
    "response_id",
    "session_id",
    "objective_uid",
    "turn_uid",
]

duplicate_selected_rows = int(
    selected_candidate_df
    .duplicated(
        subset=selected_identity
    )
    .sum()
)

print("\n" + "=" * 80)
print("SELECTED EVIDENCE IDENTITY")
print("=" * 80)

print(
    "Duplicate selected identities:",
    f"{duplicate_selected_rows:,}",
)

assert (
    duplicate_selected_rows
    ==
    0
), (
    "Duplicate selected evidence identity detected."
)

print(
    "Selected identity : PASS"
)


# ==============================================================================
# 16. SESSION LOCALITY CONTRACT
# ==============================================================================

# Every selected candidate must retain the same session_id as its
# response-objective group. Since the response itself is not yet loaded
# here, the contract is enforced against the immutable group key inherited
# from the frozen Cross-Encoder artifact.

locality_violations = 0

# Cross-Encoder rows already carry response session identity. Verify that
# each selected row has a non-null session and is internally consistent
# within its response/objective group.

group_session_counts = (
    selected_candidate_df
    .groupby(
        [
            "response_id",
            "objective_uid",
        ],
        dropna=False,
    )[
        "session_id"
    ]
    .nunique()
)

locality_violations = int(
    (
        group_session_counts
        >
        1
    ).sum()
)

print("\n" + "=" * 80)
print("SESSION LOCALITY")
print("=" * 80)

print(
    "Groups with multiple session IDs:",
    f"{locality_violations:,}",
)

assert (
    locality_violations
    ==
    0
), (
    "Session-locality contract violated."
)

print(
    "Session locality : PASS"
)


# ==============================================================================
# 17. SAVE POLICY AUDIT ARTIFACTS
# ==============================================================================

POLICY_AUDIT_ROOT = (
    EVIDENCE_PACK_AUDIT_ROOT
    / "selection_policy"
)

POLICY_AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SELECTION_DIAGNOSTICS_PATH = (
    POLICY_AUDIT_ROOT
    / "selection_diagnostics.parquet"
)

SELECTED_CANDIDATE_AUDIT_PATH = (
    POLICY_AUDIT_ROOT
    / "selected_candidate_audit.parquet"
)

selection_diagnostics_df.to_parquet(
    SELECTION_DIAGNOSTICS_PATH,
    index=False,
)

selected_candidate_df.to_parquet(
    SELECTED_CANDIDATE_AUDIT_PATH,
    index=False,
)


# ==============================================================================
# 18. CELL 3 STATUS
# ==============================================================================

EVIDENCE_PACK_CELL_3_READY = True

print("\n" + "=" * 80)
print(
    "EVIDENCE PACK CELL 3 — "
    "SELECTION POLICY + CONTRACT AUDIT: PASS"
)
print("=" * 80)

print(
    "Selection diagnostics:",
    SELECTION_DIAGNOSTICS_PATH,
)

print(
    "Selected candidate audit:",
    SELECTED_CANDIDATE_AUDIT_PATH,
)

print(
    "Policy status:",
    "DIAGNOSTICALLY VALIDATED"
)


# ==============================================================================
# 19. MEMORY CLEANUP
# ==============================================================================

for _name in [
    "ce_pf",
    "candidate_window_df",
    "selected_candidate_df",
    "selection_diagnostics_df",
    "role_counts",
    "group_session_counts",
    "scanner",
]:
    if _name in globals():
        del globals()[_name]

gc.collect()

print(
    "Cell 3 memory cleanup: PASS"
)

TRACE THE ACE — EVIDENCE PACK BUILDER
CELL 3 — EVIDENCE SELECTION POLICY + SELECTION CONTRACT AUDIT

DEPENDENCY GATE
Cell 2 dependency : PASS

FROZEN INPUT
Cross-Encoder rows: 2,482,137

SELECTION POLICY
Rank window: 16
Student top-k: 8
Tutor top-k: 4
Neighbour radius: 1
Final student window: 5
Maximum selected turns: 24

RANK WINDOW
Processed rows: 2,482,137
Window rows: 561,150
Rank-window extraction : PASS

ROLE NORMALIZATION
tutor               : 335,871
student             : 205,189
background          : 20,090
Role normalization : PASS

IDENTITY CONTRACT
Identity columns : PASS
Turn index        : PASS
Cross-Encoder score: PASS

SELECTION GROUP COVERAGE
Expected response-objective groups: 35,072
Observed groups: 35,072
Group coverage : PASS

EMPTY SELECTION AUDIT
Empty selections: 0
Empty selection audit : PASS

STUDENT EVIDENCE COVERAGE
Groups without selected student evidence: 356
Total selected student turns: 192,584
Student coverage diagnostic : RECORDED

TUTOR / CONTEXT COVE

In [15]:
# ==============================================================================
# TRACE THE ACE — EVIDENCE PACK BUILDER
# CELL 4 — TEMPORAL EVIDENCE CONSTRUCTION
# ==============================================================================

import gc
import json
import sqlite3

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq


print("=" * 80)
print("TRACE THE ACE — EVIDENCE PACK BUILDER")
print("CELL 4 — TEMPORAL EVIDENCE CONSTRUCTION")
print("=" * 80)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    "EVIDENCE_PACK_CELL_3_READY" in globals()
    and EVIDENCE_PACK_CELL_3_READY is True
), (
    "Cell 3 dependency failed."
)

print("\n" + "=" * 80)
print("DEPENDENCY GATE")
print("=" * 80)

print(
    "Cell 3 dependency : PASS"
)


# ==============================================================================
# 2. LOAD CELL 3 SELECTION AUDIT
# ==============================================================================

assert (
    SELECTED_CANDIDATE_AUDIT_PATH.exists()
), (
    "Cell 3 selected-candidate audit is missing:\n"
    f"{SELECTED_CANDIDATE_AUDIT_PATH}"
)

selected_candidates = pd.read_parquet(
    SELECTED_CANDIDATE_AUDIT_PATH
)

print("\n" + "=" * 80)
print("SELECTED CANDIDATE INPUT")
print("=" * 80)

print(
    "Rows:",
    f"{len(selected_candidates):,}",
)

print(
    "Columns:",
    selected_candidates.columns.tolist(),
)


# ==============================================================================
# 3. SELECTION INPUT CONTRACT
# ==============================================================================

REQUIRED_SELECTION_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "cross_encoder_score",
    "cross_encoder_rank",
    "canonical_role",
    "selection_source",
]

missing_columns = sorted(
    set(REQUIRED_SELECTION_COLUMNS)
    -
    set(selected_candidates.columns)
)

assert not missing_columns, (
    "Selected candidate audit is missing required columns:\n"
    f"{missing_columns}"
)

print(
    "Selection schema : PASS"
)


# ==============================================================================
# 4. AUTHORITATIVE R0 TEMPORAL SOURCE
# ==============================================================================

R0_RETRIEVAL_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "02_retrieval"
    / "R0_input"
)

SESSION_TURN_INDEX_PATH = (
    R0_RETRIEVAL_ROOT
    / "session_turn_index.parquet"
)

assert SESSION_TURN_INDEX_PATH.exists(), (
    "R0 session-turn index is missing:\n"
    f"{SESSION_TURN_INDEX_PATH}"
)

print("\n" + "=" * 80)
print("AUTHORITATIVE TEMPORAL INPUT")
print("=" * 80)

print(
    "R0 session-turn index:",
    SESSION_TURN_INDEX_PATH,
)


# ==============================================================================
# 5. R0 SCHEMA CONTRACT
# ==============================================================================

turn_pf = pq.ParquetFile(
    SESSION_TURN_INDEX_PATH
)

TURN_INDEX_COLUMNS = (
    turn_pf.schema_arrow.names
)

REQUIRED_TURN_COLUMNS = [
    "session_id",
    "turn_uid",
    "turn_index",
    "role",
    "text_norm",
]

missing_turn_columns = sorted(
    set(REQUIRED_TURN_COLUMNS)
    -
    set(TURN_INDEX_COLUMNS)
)

assert not missing_turn_columns, (
    "R0 session-turn index is missing required columns:\n"
    f"{missing_turn_columns}"
)

print(
    "R0 turn-index columns:",
    TURN_INDEX_COLUMNS,
)

print(
    "R0 temporal schema : PASS"
)


# ==============================================================================
# 6. TEMPORAL CONFIGURATION
# ==============================================================================

TEMPORAL_NEIGHBOUR_RADIUS = int(
    NEIGHBOUR_RADIUS
)

FINAL_STUDENT_ENABLED = True

FINAL_STUDENT_WINDOW = int(
    FINAL_STUDENT_WINDOW
)

assert (
    TEMPORAL_NEIGHBOUR_RADIUS >= 0
)

assert (
    FINAL_STUDENT_WINDOW >= 0
)

print("\n" + "=" * 80)
print("TEMPORAL POLICY")
print("=" * 80)

print(
    "Neighbour radius:",
    TEMPORAL_NEIGHBOUR_RADIUS,
)

print(
    "Final student enabled:",
    FINAL_STUDENT_ENABLED,
)

print(
    "Final student window:",
    FINAL_STUDENT_WINDOW,
)


# ==============================================================================
# 7. MEMORY-SAFE R0 DATASET
# ==============================================================================

R0_TURN_COLUMNS = [
    "session_id",
    "turn_uid",
    "turn_index",
    "role",
    "text_norm",
]

turn_dataset = ds.dataset(
    SESSION_TURN_INDEX_PATH,
    format="parquet",
)

print(
    "R0 Arrow dataset : PASS"
)


# ==============================================================================
# 8. BUILD REQUIRED SESSION SET
# ==============================================================================

required_sessions = (
    selected_candidates[
        "session_id"
    ]
    .astype(str)
    .drop_duplicates()
    .tolist()
)

required_session_set = set(
    required_sessions
)

print("\n" + "=" * 80)
print("TEMPORAL SESSION SCOPE")
print("=" * 80)

print(
    "Required sessions:",
    f"{len(required_session_set):,}",
)


# ==============================================================================
# 9. SINGLE-PASS R0 TEMPORAL EXTRACTION
# ==============================================================================

# IMPORTANT:
# Do NOT execute one Parquet query per selected turn.
#
# We scan the authoritative R0 session-turn index once and retain only
# sessions needed by the selected evidence groups.
#
# This avoids thousands of repeated Parquet scans.

session_turn_rows = []

scanner = turn_dataset.scanner(
    columns=R0_TURN_COLUMNS,
    batch_size=250_000,
)

r0_rows_scanned = 0
r0_rows_retained = 0

for batch in scanner.to_batches():

    batch_df = batch.to_pandas()

    r0_rows_scanned += len(
        batch_df
    )

    batch_df[
        "session_id"
    ] = (
        batch_df[
            "session_id"
        ]
        .astype(str)
    )

    relevant = batch_df[
        batch_df[
            "session_id"
        ].isin(
            required_session_set
        )
    ]

    if not relevant.empty:

        session_turn_rows.append(
            relevant
        )

        r0_rows_retained += len(
            relevant
        )

    del batch_df
    del relevant

    if (
        r0_rows_scanned
        %
        1_000_000
        ==
        0
    ):
        print(
            "R0 rows scanned:",
            f"{r0_rows_scanned:,}",
            "| retained:",
            f"{r0_rows_retained:,}",
        )

    gc.collect()


assert session_turn_rows, (
    "No R0 temporal rows were retained."
)

r0_turn_df = pd.concat(
    session_turn_rows,
    ignore_index=True,
)

del session_turn_rows
del scanner

gc.collect()


print("\n" + "=" * 80)
print("R0 TEMPORAL EXTRACTION")
print("=" * 80)

print(
    "Rows scanned:",
    f"{r0_rows_scanned:,}",
)

print(
    "Rows retained:",
    f"{r0_rows_retained:,}",
)

print(
    "Retained sessions:",
    f"{r0_turn_df['session_id'].nunique():,}",
)

assert (
    r0_turn_df[
        "session_id"
    ]
    .nunique()
    ==
    len(required_session_set)
), (
    "Not all required sessions were found in R0."
)

print(
    "R0 temporal extraction : PASS"
)


# ==============================================================================
# 10. R0 TURN IDENTITY CONTRACT
# ==============================================================================

assert (
    r0_turn_df[
        "turn_uid"
    ]
    .notna()
    .all()
)

assert (
    r0_turn_df[
        "turn_index"
    ]
    .notna()
    .all()
)

assert (
    r0_turn_df[
        "role"
    ]
    .notna()
    .all()
)

print(
    "R0 turn identity : PASS"
)


# ==============================================================================
# 11. FINAL ROLE NORMALIZATION
# ==============================================================================

def canonical_role_local(value):

    if pd.isna(value):
        return "unknown"

    value = (
        str(value)
        .strip()
        .lower()
    )

    if value in {
        "student",
        "learner",
        "user",
    }:
        return "student"

    if value in {
        "tutor",
        "teacher",
        "assistant",
    }:
        return "tutor"

    return value


r0_turn_df[
    "canonical_role"
] = (
    r0_turn_df[
        "role"
    ]
    .map(
        canonical_role_local
    )
)

print(
    "R0 canonical role normalization : PASS"
)


# ==============================================================================
# 12. BUILD SESSION INDEX
# ==============================================================================

r0_turn_df = (
    r0_turn_df
    .sort_values(
        [
            "session_id",
            "turn_index",
            "turn_uid",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

session_groups = {
    str(session_id): group.reset_index(
        drop=True
    )
    for session_id, group in (
        r0_turn_df
        .groupby(
            "session_id",
            sort=False,
        )
    )
}

print(
    "Session temporal index : PASS"
)

# ==============================================================================
# 13. TEMPORAL EVIDENCE CONSTRUCTION
# ==============================================================================

temporal_rows = []

groups_processed = 0

group_columns = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
]

unique_group_count = (
    selected_candidates[
        group_columns
    ]
    .drop_duplicates()
    .shape[0]
)

print("\n" + "=" * 80)
print("TEMPORAL CONSTRUCTION")
print("=" * 80)

print(
    "Response-objective groups:",
    f"{unique_group_count:,}",
)

assert (
    unique_group_count
    ==
    35_072
), (
    "Unexpected number of response-objective groups."
)


for (
    response_id,
    session_id,
    objective_uid,
    fold,
), group in selected_candidates.groupby(
    group_columns,
    sort=False,
):

    groups_processed += 1

    session_id = str(
        session_id
    )

    assert (
        session_id
        in
        session_groups
    ), (
        "Selected evidence session missing "
        "from R0 temporal index."
    )

    session_df = session_groups[
        session_id
    ]

    evidence_map = {}

    # --------------------------------------------------------------------------
    # A. CROSS-ENCODER SELECTED TURNS
    # --------------------------------------------------------------------------

    for _, row in group.iterrows():

        turn_uid = str(
            row["turn_uid"]
        )

        evidence_map[
            turn_uid
        ] = {
            "response_id": str(
                response_id
            ),
            "session_id": session_id,
            "objective_uid": str(
                objective_uid
            ),
            "fold": int(
                fold
            ),
            "turn_uid": turn_uid,
            "turn_index": int(
                row["turn_index"]
            ),
            "role": str(
                row["role"]
            ),
            "canonical_role": str(
                row["canonical_role"]
            ),
            "text_norm": None,
            "evidence_source": (
                "cross_encoder_selected"
            ),
            "cross_encoder_rank": int(
                row["cross_encoder_rank"]
            ),
            "cross_encoder_score": float(
                row["cross_encoder_score"]
            ),
            "selection_priority": 1,
        }

    # --------------------------------------------------------------------------
    # B. TEMPORAL NEIGHBOURS
    # --------------------------------------------------------------------------

    selected_indices = (
        group[
            "turn_index"
        ]
        .astype(int)
        .tolist()
    )

    for center_index in selected_indices:

        lower_index = max(
            0,
            center_index
            -
            TEMPORAL_NEIGHBOUR_RADIUS,
        )

        upper_index = (
            center_index
            +
            TEMPORAL_NEIGHBOUR_RADIUS
        )

        neighbour_df = (
            session_df[
                (
                    session_df[
                        "turn_index"
                    ]
                    >=
                    lower_index
                )
                &
                (
                    session_df[
                        "turn_index"
                    ]
                    <=
                    upper_index
                )
            ]
        )

        for _, neighbour in (
            neighbour_df.iterrows()
        ):

            turn_uid = str(
                neighbour["turn_uid"]
            )

            if (
                turn_uid
                in
                evidence_map
            ):
                continue

            evidence_map[
                turn_uid
            ] = {
                "response_id": str(
                    response_id
                ),
                "session_id": session_id,
                "objective_uid": str(
                    objective_uid
                ),
                "fold": int(
                    fold
                ),
                "turn_uid": turn_uid,
                "turn_index": int(
                    neighbour[
                        "turn_index"
                    ]
                ),
                "role": str(
                    neighbour[
                        "role"
                    ]
                ),
                "canonical_role": str(
                    neighbour[
                        "canonical_role"
                    ]
                ),
                "text_norm": (
                    None
                    if pd.isna(
                        neighbour[
                            "text_norm"
                        ]
                    )
                    else str(
                        neighbour[
                            "text_norm"
                        ]
                    )
                ),
                "evidence_source": (
                    "temporal_neighbour"
                ),
                "cross_encoder_rank": None,
                "cross_encoder_score": None,
                "selection_priority": 2,
            }

    # --------------------------------------------------------------------------
    # C. FINAL STUDENT TURNS
    # --------------------------------------------------------------------------

    if FINAL_STUDENT_ENABLED:

        final_students = (
            session_df[
                session_df[
                    "canonical_role"
                ]
                ==
                "student"
            ]
            .sort_values(
                [
                    "turn_index",
                    "turn_uid",
                ],
                kind="mergesort",
            )
            .tail(
                FINAL_STUDENT_WINDOW
            )
        )

        for _, student_row in (
            final_students.iterrows()
        ):

            turn_uid = str(
                student_row[
                    "turn_uid"
                ]
            )

            if (
                turn_uid
                in
                evidence_map
            ):

                evidence_map[
                    turn_uid
                ][
                    "evidence_source"
                ] = (
                    evidence_map[
                        turn_uid
                    ][
                        "evidence_source"
                    ]
                    +
                    "+final_student"
                )

                continue

            evidence_map[
                turn_uid
            ] = {
                "response_id": str(
                    response_id
                ),
                "session_id": session_id,
                "objective_uid": str(
                    objective_uid
                ),
                "fold": int(
                    fold
                ),
                "turn_uid": turn_uid,
                "turn_index": int(
                    student_row[
                        "turn_index"
                    ]
                ),
                "role": str(
                    student_row[
                        "role"
                    ]
                ),
                "canonical_role": "student",
                "text_norm": (
                    None
                    if pd.isna(
                        student_row[
                            "text_norm"
                        ]
                    )
                    else str(
                        student_row[
                            "text_norm"
                        ]
                    )
                ),
                "evidence_source": (
                    "final_student"
                ),
                "cross_encoder_rank": None,
                "cross_encoder_score": None,
                "selection_priority": 3,
            }

    # --------------------------------------------------------------------------
    # D. MATERIALIZE
    # --------------------------------------------------------------------------

    temporal_rows.extend(
        evidence_map.values()
    )

    if groups_processed % 1000 == 0:

        print(
            "Groups processed:",
            f"{groups_processed:,}",
        )


# ==============================================================================
# 14. MATERIALIZE TEMPORAL EVIDENCE
# ==============================================================================

temporal_evidence_df = pd.DataFrame(
    temporal_rows
)

del temporal_rows

gc.collect()

assert (
    not temporal_evidence_df.empty
), (
    "Temporal evidence construction produced no rows."
)

temporal_group_count = (
    temporal_evidence_df[
        [
            "response_id",
            "objective_uid",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print("\n" + "=" * 80)
print("TEMPORAL EVIDENCE RESULT")
print("=" * 80)

print(
    "Rows:",
    f"{len(temporal_evidence_df):,}",
)

print(
    "Groups:",
    f"{temporal_group_count:,}",
)


# ==============================================================================
# 15. GROUP COVERAGE CONTRACT
# ==============================================================================

assert (
    temporal_group_count
    ==
    35_072
), (
    "Temporal evidence does not cover all "
    "response-objective groups."
)

print(
    "Response-objective coverage : PASS"
)


# ==============================================================================
# 16. SESSION LOCALITY
# ==============================================================================

session_counts = (
    temporal_evidence_df
    .groupby(
        [
            "response_id",
            "objective_uid",
        ],
        dropna=False,
    )[
        "session_id"
    ]
    .nunique()
)

session_violations = int(
    (
        session_counts
        >
        1
    ).sum()
)

assert (
    session_violations
    ==
    0
), (
    "Temporal evidence crosses session boundaries."
)

print(
    "Session locality : PASS"
)


# ==============================================================================
# 17. IDENTITY DEDUPLICATION
# ==============================================================================

IDENTITY_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "turn_uid",
]

duplicate_count = int(
    temporal_evidence_df
    .duplicated(
        subset=IDENTITY_COLUMNS
    )
    .sum()
)

assert (
    duplicate_count
    ==
    0
), (
    "Duplicate temporal evidence identities detected."
)

print(
    "Duplicate identity : PASS"
)


# ==============================================================================
# 18. CHRONOLOGICAL ORDER
# ==============================================================================

temporal_evidence_df = (
    temporal_evidence_df
    .sort_values(
        [
            "response_id",
            "objective_uid",
            "turn_index",
            "turn_uid",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

print(
    "Chronological ordering : PASS"
)


# ==============================================================================
# 19. EVIDENCE SOURCE DISTRIBUTION
# ==============================================================================

source_distribution = (
    temporal_evidence_df[
        "evidence_source"
    ]
    .value_counts(
        dropna=False
    )
)

print("\n" + "=" * 80)
print("EVIDENCE SOURCE DISTRIBUTION")
print("=" * 80)

for source, count in (
    source_distribution.items()
):

    print(
        f"{str(source):40s}:",
        f"{int(count):,}",
    )


# ==============================================================================
# 20. ROLE DISTRIBUTION
# ==============================================================================

temporal_role_distribution = (
    temporal_evidence_df[
        "canonical_role"
    ]
    .value_counts(
        dropna=False
    )
)

print("\n" + "=" * 80)
print("TEMPORAL EVIDENCE ROLE DISTRIBUTION")
print("=" * 80)

for role, count in (
    temporal_role_distribution.items()
):

    print(
        f"{str(role):20s}:",
        f"{int(count):,}",
    )


# ==============================================================================
# 21. PER-GROUP EVIDENCE SIZE
# ==============================================================================

group_size_df = (
    temporal_evidence_df
    .groupby(
        [
            "response_id",
            "objective_uid",
        ],
        as_index=False,
    )
    .agg(
        evidence_turn_count=(
            "turn_uid",
            "nunique",
        ),
        min_turn_index=(
            "turn_index",
            "min",
        ),
        max_turn_index=(
            "turn_index",
            "max",
        ),
    )
)

assert (
    group_size_df[
        "evidence_turn_count"
    ]
    .min()
    >=
    1
), (
    "A response-objective group has zero evidence turns."
)

print("\n" + "=" * 80)
print("EVIDENCE SIZE")
print("=" * 80)

print(
    "Min:",
    int(
        group_size_df[
            "evidence_turn_count"
        ].min()
    ),
)

print(
    "Max:",
    int(
        group_size_df[
            "evidence_turn_count"
        ].max()
    ),
)

print(
    "Mean:",
    float(
        group_size_df[
            "evidence_turn_count"
        ].mean()
    ),
)

print(
    "Median:",
    float(
        group_size_df[
            "evidence_turn_count"
        ].median()
    ),
)


# ==============================================================================
# 22. SAVE TEMPORAL AUDITS
# ==============================================================================

TEMPORAL_AUDIT_ROOT = (
    EVIDENCE_PACK_AUDIT_ROOT
    / "temporal"
)

TEMPORAL_AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

TEMPORAL_EVIDENCE_AUDIT_PATH = (
    TEMPORAL_AUDIT_ROOT
    / "temporal_evidence_candidates.parquet"
)

TEMPORAL_GROUP_AUDIT_PATH = (
    TEMPORAL_AUDIT_ROOT
    / "temporal_group_distribution.parquet"
)

temporal_evidence_df.to_parquet(
    TEMPORAL_EVIDENCE_AUDIT_PATH,
    index=False,
)

group_size_df.to_parquet(
    TEMPORAL_GROUP_AUDIT_PATH,
    index=False,
)


# ==============================================================================
# 23. FINAL CELL GATE
# ==============================================================================

EVIDENCE_PACK_CELL_4_READY = True

print("\n" + "=" * 80)
print(
    "EVIDENCE PACK CELL 4 — "
    "TEMPORAL EVIDENCE CONSTRUCTION: PASS"
)
print("=" * 80)

print(
    "Temporal evidence audit:",
    TEMPORAL_EVIDENCE_AUDIT_PATH,
)

print(
    "Temporal group audit:",
    TEMPORAL_GROUP_AUDIT_PATH,
)

print(
    "Session locality : PASS"
)

print(
    "Identity deduplication : PASS"
)

print(
    "Chronological ordering : PASS"
)


# ==============================================================================
# 24. MEMORY CLEANUP
# ==============================================================================

for _name in [
    "selected_candidates",
    "r0_turn_df",
    "session_groups",
    "group_size_df",
    "session_counts",
    "source_distribution",
    "temporal_role_distribution",
    "turn_pf",
    "turn_dataset",
    "scanner",
]:
    if _name in globals():
        del globals()[_name]

gc.collect()

print(
    "Cell 4 memory cleanup: PASS"
)

TRACE THE ACE — EVIDENCE PACK BUILDER
CELL 4 — TEMPORAL EVIDENCE CONSTRUCTION

DEPENDENCY GATE
Cell 3 dependency : PASS

SELECTED CANDIDATE INPUT
Rows: 332,087
Columns: ['response_id', 'session_id', 'objective_uid', 'fold', 'turn_uid', 'role', 'turn_index', 'cross_encoder_score', 'cross_encoder_rank', 'canonical_role', 'selection_source']
Selection schema : PASS

AUTHORITATIVE TEMPORAL INPUT
R0 session-turn index: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input\session_turn_index.parquet
R0 turn-index columns: ['session_id', 'turn_uid', 'turn_index', 'role', 'text_norm', 'relative_turn_position', 'previous_role', 'next_role', 'speaker_switch', 'time_since_previous_turn', 'elapsed_from_session_start', 'ordering_method', 'ordering_confidence', 'ordering_comparability']
R0 temporal schema : PASS

TEMPORAL POLICY
Neighbour radius: 1
Final student enabled: True
Final student window: 5
R0 Arrow dataset : PASS

TEMPORAL SESSION SCOPE
Required sessions: 22,821

In [17]:
# ==============================================================================
# TRACE THE ACE — EVIDENCE PACK BUILDER
# CELL 5 — 2048-TOKEN EVIDENCE PACK SERIALIZATION
# ==============================================================================

import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq

from transformers import AutoTokenizer


print("=" * 90)
print("TRACE THE ACE — EVIDENCE PACK BUILDER")
print("CELL 5 — 2048-TOKEN EVIDENCE PACK SERIALIZATION")
print("=" * 90)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    "EVIDENCE_PACK_CELL_4_READY" in globals()
    and EVIDENCE_PACK_CELL_4_READY is True
), (
    "Cell 4 dependency failed."
)

print("\n" + "=" * 90)
print("DEPENDENCY GATE")
print("=" * 90)

print(
    "Cell 4 dependency : PASS"
)


# ==============================================================================
# 2. PROJECT PATHS
# ==============================================================================

PROJECT_ROOT = Path(
    PROJECT_ROOT
)

SCRATCH_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
)

RETRIEVAL_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
)

R0_ROOT = (
    RETRIEVAL_ROOT
    / "R0_input"
)

EVIDENCE_ROOT = (
    SCRATCH_ROOT
    / "03_evidence_pack"
)

EVIDENCE_AUDIT_ROOT = (
    EVIDENCE_ROOT
    / "audit"
)

EVIDENCE_PACK_ROOT = (
    EVIDENCE_ROOT
    / "packs"
)

TEMPORAL_AUDIT_ROOT = (
    EVIDENCE_AUDIT_ROOT
    / "temporal"
)

SERIALIZATION_AUDIT_ROOT = (
    EVIDENCE_AUDIT_ROOT
    / "serialization"
)

EVIDENCE_AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

EVIDENCE_PACK_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SERIALIZATION_AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ==============================================================================
# 3. INPUT ARTIFACTS
# ==============================================================================

TEMPORAL_EVIDENCE_PATH = (
    TEMPORAL_AUDIT_ROOT
    / "temporal_evidence_candidates.parquet"
)

OBJECTIVE_CATALOGUE_PATH = (
    R0_ROOT
    / "objective_catalogue.parquet"
)

SESSION_TURN_INDEX_PATH = (
    R0_ROOT
    / "session_turn_index.parquet"
)


assert (
    TEMPORAL_EVIDENCE_PATH.exists()
), (
    "Cell 4 temporal evidence artifact is missing:\n"
    f"{TEMPORAL_EVIDENCE_PATH}"
)

assert (
    OBJECTIVE_CATALOGUE_PATH.exists()
), (
    "Objective catalogue is missing:\n"
    f"{OBJECTIVE_CATALOGUE_PATH}"
)

assert (
    SESSION_TURN_INDEX_PATH.exists()
), (
    "R0 session-turn index is missing:\n"
    f"{SESSION_TURN_INDEX_PATH}"
)


print("\n" + "=" * 90)
print("INPUT ARTIFACTS")
print("=" * 90)

print(
    "Temporal evidence:",
    TEMPORAL_EVIDENCE_PATH,
)

print(
    "Objective catalogue:",
    OBJECTIVE_CATALOGUE_PATH,
)

print(
    "R0 session-turn index:",
    SESSION_TURN_INDEX_PATH,
)


# ==============================================================================
# 4. LOAD TEMPORAL EVIDENCE
# ==============================================================================

temporal_evidence_df = pd.read_parquet(
    TEMPORAL_EVIDENCE_PATH
)

assert (
    not temporal_evidence_df.empty
), (
    "Temporal evidence artifact is empty."
)

print("\n" + "=" * 90)
print("TEMPORAL EVIDENCE INPUT")
print("=" * 90)

print(
    "Rows:",
    f"{len(temporal_evidence_df):,}",
)

print(
    "Columns:",
    temporal_evidence_df.columns.tolist(),
)


# ==============================================================================
# 5. TEMPORAL INPUT CONTRACT
# ==============================================================================

REQUIRED_TEMPORAL_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "turn_index",
    "role",
    "canonical_role",
    "text_norm",
    "evidence_source",
    "cross_encoder_rank",
    "cross_encoder_score",
]

missing_temporal_columns = sorted(
    set(REQUIRED_TEMPORAL_COLUMNS)
    -
    set(temporal_evidence_df.columns)
)

assert not missing_temporal_columns, (
    "Temporal evidence is missing required columns:\n"
    f"{missing_temporal_columns}"
)

print(
    "Temporal evidence schema : PASS"
)


# ==============================================================================
# 6. LOAD OBJECTIVE CATALOGUE
# ==============================================================================

objective_catalogue = pd.read_parquet(
    OBJECTIVE_CATALOGUE_PATH
)

assert (
    "objective_uid"
    in
    objective_catalogue.columns
), (
    "Objective catalogue missing objective_uid."
)

OBJECTIVE_TEXT_COLUMN = None

for candidate_column in [
    "objective_raw",
    "objective_text",
    "objective",
]:

    if (
        candidate_column
        in
        objective_catalogue.columns
    ):

        OBJECTIVE_TEXT_COLUMN = (
            candidate_column
        )

        break


assert (
    OBJECTIVE_TEXT_COLUMN is not None
), (
    "Could not identify objective text column."
)


print("\n" + "=" * 90)
print("OBJECTIVE CATALOGUE")
print("=" * 90)

print(
    "Rows:",
    f"{len(objective_catalogue):,}",
)

print(
    "Objective text column:",
    OBJECTIVE_TEXT_COLUMN,
)


# ==============================================================================
# 7. OBJECTIVE MAP
# ==============================================================================

objective_catalogue[
    "objective_uid"
] = (
    objective_catalogue[
        "objective_uid"
    ]
    .astype(str)
)

objective_map = (
    objective_catalogue[
        [
            "objective_uid",
            OBJECTIVE_TEXT_COLUMN,
        ]
    ]
    .drop_duplicates(
        subset=[
            "objective_uid"
        ]
    )
    .set_index(
        "objective_uid"
    )[
        OBJECTIVE_TEXT_COLUMN
    ]
    .astype(str)
    .to_dict()
)

assert (
    len(objective_map)
    ==
    objective_catalogue[
        "objective_uid"
    ].nunique()
), (
    "Objective map construction failed."
)

print(
    "Objective map : PASS"
)


# ==============================================================================
# 8. DISCOVER CANONICAL RESPONSE ARTIFACT
# ==============================================================================

# Prefer an existing canonical-response path if Cell 0 already exposed one.
candidate_response_paths = []

for variable_name in [
    "CANONICAL_RESPONSES_PATH",
    "RESPONSES_PATH",
    "CANONICAL_RESPONSE_PATH",
]:

    if (
        variable_name
        in
        globals()
    ):

        candidate_value = globals()[
            variable_name
        ]

        if candidate_value is not None:

            candidate_response_paths.append(
                Path(
                    candidate_value
                )
            )


candidate_response_paths.extend(
    [
        (
            SCRATCH_ROOT
            / "01_data_foundation"
            / "03_integrity"
            / "canonical"
            / "responses.parquet"
        ),
        (
            SCRATCH_ROOT
            / "01_data_foundation"
            / "canonical"
            / "responses.parquet"
        ),
        (
            SCRATCH_ROOT
            / "01_data_foundation"
            / "responses.parquet"
        ),
    ]
)


RESPONSE_ARTIFACT_PATH = None

for candidate_path in candidate_response_paths:

    if candidate_path.exists():

        RESPONSE_ARTIFACT_PATH = (
            candidate_path
        )

        break


assert (
    RESPONSE_ARTIFACT_PATH is not None
), (
    "Could not locate canonical responses parquet.\n"
    "Checked:\n"
    +
    "\n".join(
        str(path)
        for path in candidate_response_paths
    )
)


print("\n" + "=" * 90)
print("CANONICAL RESPONSE ARTIFACT")
print("=" * 90)

print(
    "Response artifact:",
    RESPONSE_ARTIFACT_PATH,
)


# ==============================================================================
# 9. RESPONSE SCHEMA DISCOVERY
# ==============================================================================

response_pf = pq.ParquetFile(
    RESPONSE_ARTIFACT_PATH
)

response_columns = (
    response_pf.schema_arrow.names
)

REQUIRED_RESPONSE_ID_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
]

missing_response_id_columns = sorted(
    set(REQUIRED_RESPONSE_ID_COLUMNS)
    -
    set(response_columns)
)

assert not missing_response_id_columns, (
    "Canonical response artifact missing:\n"
    f"{missing_response_id_columns}"
)


# ==============================================================================
# 10. TARGET COLUMN DISCOVERY
# ==============================================================================

TARGET_COLUMN = None

for candidate_column in [
    "target",
    "label",
    "is_correct",
    "y",
]:

    if (
        candidate_column
        in
        response_columns
    ):

        TARGET_COLUMN = (
            candidate_column
        )

        break


assert (
    TARGET_COLUMN is not None
), (
    "Could not identify canonical target column."
)


print(
    "Target column:",
    TARGET_COLUMN,
)


# ==============================================================================
# 11. LOAD ONLY REQUIRED RESPONSE METADATA
# ==============================================================================

response_meta = pd.read_parquet(
    RESPONSE_ARTIFACT_PATH,
    columns=[
        "response_id",
        "session_id",
        "objective_uid",
        "fold",
        TARGET_COLUMN,
    ],
)

response_meta[
    "response_id"
] = (
    response_meta[
        "response_id"
    ]
    .astype(str)
)

response_meta[
    "session_id"
] = (
    response_meta[
        "session_id"
    ]
    .astype(str)
)

response_meta[
    "objective_uid"
] = (
    response_meta[
        "objective_uid"
    ]
    .astype(str)
)


assert (
    response_meta[
        "response_id"
    ].is_unique
), (
    "Canonical response_id is not unique."
)

assert (
    response_meta[
        TARGET_COLUMN
    ].notna()
    .all()
), (
    "Canonical target contains nulls."
)

print(
    "Canonical response metadata : PASS"
)


# ==============================================================================
# 12. CROSS-CHECK RESPONSE POPULATION
# ==============================================================================

TEMPORAL_RESPONSE_COUNT = (
    temporal_evidence_df[
        "response_id"
    ]
    .astype(str)
    .nunique()
)

CANONICAL_RESPONSE_COUNT = (
    response_meta[
        "response_id"
    ]
    .nunique()
)

print("\n" + "=" * 90)
print("RESPONSE POPULATION")
print("=" * 90)

print(
    "Temporal evidence responses:",
    f"{TEMPORAL_RESPONSE_COUNT:,}",
)

print(
    "Canonical responses:",
    f"{CANONICAL_RESPONSE_COUNT:,}",
)

assert (
    TEMPORAL_RESPONSE_COUNT
    ==
    35_072
), (
    "Temporal evidence does not cover 35,072 responses."
)

assert (
    CANONICAL_RESPONSE_COUNT
    ==
    35_072
), (
    "Canonical response population is not 35,072."
)

print(
    "Response population : PASS"
)


# ==============================================================================
# 13. REPAIR MISSING TEXT
# ==============================================================================

missing_text_mask = (
    temporal_evidence_df[
        "text_norm"
    ]
    .isna()
)

missing_text_uids = set(
    temporal_evidence_df.loc[
        missing_text_mask,
        "turn_uid",
    ]
    .astype(str)
)

print("\n" + "=" * 90)
print("TEXT INTEGRITY / REPAIR")
print("=" * 90)

print(
    "Missing text rows:",
    f"{int(missing_text_mask.sum()):,}",
)

print(
    "Missing text turn UIDs:",
    f"{len(missing_text_uids):,}",
)


if missing_text_uids:

    r0_dataset = ds.dataset(
        SESSION_TURN_INDEX_PATH,
        format="parquet",
    )

    repair_rows = []

    scanner = r0_dataset.scanner(
        columns=[
            "turn_uid",
            "text_norm",
        ],
        batch_size=250_000,
    )

    for batch in scanner.to_batches():

        batch_df = (
            batch
            .to_pandas()
        )

        batch_df[
            "turn_uid"
        ] = (
            batch_df[
                "turn_uid"
            ]
            .astype(str)
        )

        matched = batch_df[
            batch_df[
                "turn_uid"
            ].isin(
                missing_text_uids
            )
        ]

        if not matched.empty:

            repair_rows.append(
                matched
            )

        del batch_df
        del matched

    del scanner
    del r0_dataset

    gc.collect()

    assert repair_rows, (
        "R0 text repair found no matching rows."
    )

    repair_df = pd.concat(
        repair_rows,
        ignore_index=True,
    )

    del repair_rows

    gc.collect()

    repair_df = (
        repair_df
        .drop_duplicates(
            subset=[
                "turn_uid"
            ]
        )
    )

    assert (
        len(repair_df)
        ==
        len(missing_text_uids)
    ), (
        "R0 repair did not recover all missing turn UIDs."
    )

    repair_map = dict(
        zip(
            repair_df[
                "turn_uid"
            ].astype(str),
            repair_df[
                "text_norm"
            ],
        )
    )

    temporal_evidence_df[
        "text_norm"
    ] = (
        temporal_evidence_df[
            "text_norm"
        ]
        .fillna(
            temporal_evidence_df[
                "turn_uid"
            ]
            .astype(str)
            .map(
                repair_map
            )
        )
    )

    del repair_df
    del repair_map

    gc.collect()


assert (
    temporal_evidence_df[
        "text_norm"
    ]
    .notna()
    .all()
), (
    "Evidence text still contains null values."
)

assert (
    temporal_evidence_df[
        "text_norm"
    ]
    .astype(str)
    .str.strip()
    .ne("")
    .all()
), (
    "Evidence text contains empty strings."
)

print(
    "Evidence text integrity : PASS"
)


# ==============================================================================
# 14. MODERNBERT TOKENIZER CONTRACT
# ==============================================================================

MODERNBERT_MODEL_NAME = (
    "answerdotai/ModernBERT-base"
)

MAX_EVIDENCE_TOKENS = 2048


print("\n" + "=" * 90)
print("MODERNBERT TOKENIZER")
print("=" * 90)

print(
    "Tokenizer model:",
    MODERNBERT_MODEL_NAME,
)

print(
    "Maximum input tokens:",
    MAX_EVIDENCE_TOKENS,
)

print(
    "Model weights:",
    "NOT LOADED",
)


tokenizer = AutoTokenizer.from_pretrained(
    MODERNBERT_MODEL_NAME,
    use_fast=True,
)

assert (
    tokenizer.is_fast
), (
    "Fast tokenizer is required."
)

print(
    "Tokenizer class:",
    tokenizer.__class__.__name__,
)

print(
    "Tokenizer load : PASS"
)


# ==============================================================================
# 15. EXACT TOKEN COUNT
# ==============================================================================

def token_count(
    text,
):
    """
    Count actual model input tokens, including special tokens.
    """

    if (
        text is None
        or
        str(text).strip() == ""
    ):
        return 0

    encoded = tokenizer(
        str(text),
        add_special_tokens=True,
        truncation=False,
        return_attention_mask=False,
    )

    return len(
        encoded[
            "input_ids"
        ]
    )


# ==============================================================================
# 16. HARD TOKEN TRUNCATION
# ==============================================================================

def truncate_to_tokens(
    text,
    max_tokens,
):
    """
    Deterministic tokenizer-aware truncation.

    The returned text is re-tokenized with special tokens and verified
    against the hard budget before returning.
    """

    if (
        text is None
        or
        str(text).strip() == ""
    ):
        return ""

    if (
        max_tokens
        <=
        0
    ):
        return ""

    text = str(
        text
    )

    encoded = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False,
        return_attention_mask=False,
    )

    content_ids = encoded[
        "input_ids"
    ]

    # First approximation.
    candidate_ids = content_ids[
        :max_tokens
    ]

    candidate_text = tokenizer.decode(
        candidate_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    # Exact correction loop.
    #
    # This guarantees the FINAL decoded string satisfies the model-input
    # token contract after special tokens are added.
    while (
        token_count(
            candidate_text
        )
        >
        max_tokens
    ):

        candidate_ids = candidate_ids[
            :-1
        ]

        if not candidate_ids:
            return ""

        candidate_text = tokenizer.decode(
            candidate_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

    return candidate_text


# ==============================================================================
# 17. TEXT NORMALIZATION
# ==============================================================================

def normalize_text(
    value,
):
    if pd.isna(value):
        return ""

    return str(
        value
    ).strip()


# ==============================================================================
# 18. SECTION SERIALIZER
# ==============================================================================

def make_section_text(
    section_name,
    rows,
):
    """
    Deterministic chronological serialization of evidence turns.
    """

    if rows.empty:
        return ""

    rows = (
        rows
        .sort_values(
            [
                "turn_index",
                "turn_uid",
            ],
            kind="mergesort",
        )
    )

    parts = [
        f"[{section_name}]"
    ]

    for _, row in rows.iterrows():

        role = str(
            row[
                "canonical_role"
            ]
        ).upper()

        turn_index = int(
            row[
                "turn_index"
            ]
        )

        text = normalize_text(
            row[
                "text_norm"
            ]
        )

        if not text:
            continue

        parts.append(
            f"{role} T{turn_index}: {text}"
        )

    if len(parts) <= 1:
        return ""

    return "\n".join(
        parts
    )


# ==============================================================================
# 19. EVIDENCE PACK BUILDER
# ==============================================================================

def build_pack(
    objective_text,
    group_df,
):
    """
    Build one deterministic evidence pack.

    Priority:

    1. Objective
    2. Strong student evidence
    3. Final student evidence
    4. Student after-feedback sequence
    5. Tutor context

    The final text is always re-tokenized and hard-capped at 2048 tokens.
    """

    group_df = (
        group_df
        .sort_values(
            [
                "turn_index",
                "turn_uid",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    student_df = group_df[
        group_df[
            "canonical_role"
        ]
        ==
        "student"
    ].copy()

    tutor_df = group_df[
        group_df[
            "canonical_role"
        ]
        ==
        "tutor"
    ].copy()

    final_student_mask = (
        group_df[
            "evidence_source"
        ]
        .astype(str)
        .str.contains(
            "final_student",
            regex=False,
        )
        &
        (
            group_df[
                "canonical_role"
            ]
            ==
            "student"
        )
    )

    final_student_df = (
        group_df[
            final_student_mask
        ]
        .copy()
    )

    # --------------------------------------------------------------------------
    # First tutor turn = temporal feedback boundary.
    # --------------------------------------------------------------------------

    tutor_indices = (
        tutor_df[
            "turn_index"
        ]
        .astype(int)
        .tolist()
    )

    if tutor_indices:

        first_tutor_index = min(
            tutor_indices
        )

        student_before_df = (
            student_df[
                student_df[
                    "turn_index"
                ]
                <
                first_tutor_index
            ]
            .copy()
        )

        student_after_df = (
            student_df[
                student_df[
                    "turn_index"
                ]
                >
                first_tutor_index
            ]
            .copy()
        )

    else:

        student_before_df = (
            student_df.copy()
        )

        student_after_df = (
            student_df.iloc[
                0:0
            ].copy()
        )

    # --------------------------------------------------------------------------
    # Do not serialize final student turns twice.
    # --------------------------------------------------------------------------

    final_student_uids = set(
        final_student_df[
            "turn_uid"
        ]
        .astype(str)
    )

    student_after_df = (
        student_after_df[
            ~student_after_df[
                "turn_uid"
            ]
            .astype(str)
            .isin(
                final_student_uids
            )
        ]
    )

    # --------------------------------------------------------------------------
    # Section construction.
    # --------------------------------------------------------------------------

    sections = {
        "OBJECTIVE": (
            "[OBJECTIVE]\n"
            +
            normalize_text(
                objective_text
            )
        ),

        "STUDENT_BEFORE_FEEDBACK": (
            make_section_text(
                "STUDENT_BEFORE_FEEDBACK",
                student_before_df,
            )
        ),

        "FINAL_STUDENT_EVIDENCE": (
            make_section_text(
                "FINAL_STUDENT_EVIDENCE",
                final_student_df,
            )
        ),

        "STUDENT_AFTER_FEEDBACK": (
            make_section_text(
                "STUDENT_AFTER_FEEDBACK",
                student_after_df,
            )
        ),

        "TUTOR_CONTEXT": (
            make_section_text(
                "TUTOR_CONTEXT",
                tutor_df,
            )
        ),
    }

    # --------------------------------------------------------------------------
    # Priority order.
    # --------------------------------------------------------------------------

    section_priority = [
        "OBJECTIVE",
        "STUDENT_BEFORE_FEEDBACK",
        "FINAL_STUDENT_EVIDENCE",
        "STUDENT_AFTER_FEEDBACK",
        "TUTOR_CONTEXT",
    ]

    selected_sections = []

    # Conservative incremental budget.
    current_text = ""

    for section_name in section_priority:

        section_text = sections[
            section_name
        ]

        if not section_text:
            continue

        if not current_text:

            candidate_text = (
                section_text
            )

        else:

            candidate_text = (
                current_text
                +
                "\n\n"
                +
                section_text
            )

        candidate_tokens = token_count(
            candidate_text
        )

        if (
            candidate_tokens
            <=
            MAX_EVIDENCE_TOKENS
        ):

            current_text = (
                candidate_text
            )

            selected_sections.append(
                section_name
            )

            continue

        # ----------------------------------------------------------------------
        # Section does not fit completely.
        #
        # Preserve everything already accepted.
        # Use remaining budget for this section.
        # ----------------------------------------------------------------------

        remaining_budget = (
            MAX_EVIDENCE_TOKENS
            -
            token_count(
                current_text
            )
        )

        if (
            remaining_budget
            <=
            0
        ):
            break

        prefix = (
            ""
            if not current_text
            else
            current_text
            +
            "\n\n"
        )

        prefix_tokens = token_count(
            prefix
        )

        section_budget = (
            MAX_EVIDENCE_TOKENS
            -
            prefix_tokens
        )

        if (
            section_budget
            <=
            0
        ):
            break

        truncated_section = (
            truncate_to_tokens(
                section_text,
                section_budget,
            )
        )

        if (
            truncated_section.strip()
        ):

            candidate_text = (
                prefix
                +
                truncated_section
            )

            # Final exact hard-cap.
            candidate_text = (
                truncate_to_tokens(
                    candidate_text,
                    MAX_EVIDENCE_TOKENS,
                )
            )

            current_text = (
                candidate_text
            )

            selected_sections.append(
                section_name
            )

        break

    evidence_text = (
        current_text
        .strip()
    )

    # --------------------------------------------------------------------------
    # Absolute final safety gate.
    # --------------------------------------------------------------------------

    actual_tokens = token_count(
        evidence_text
    )

    if (
        actual_tokens
        >
        MAX_EVIDENCE_TOKENS
    ):

        evidence_text = (
            truncate_to_tokens(
                evidence_text,
                MAX_EVIDENCE_TOKENS,
            )
        )

        actual_tokens = token_count(
            evidence_text
        )

    assert (
        actual_tokens
        <=
        MAX_EVIDENCE_TOKENS
    ), (
        "Serialized evidence exceeds 2048 tokens."
    )

    assert (
        evidence_text.strip()
    ), (
        "Empty evidence pack generated."
    )

    return (
        evidence_text,
        int(actual_tokens),
        sections,
        selected_sections,
    )


# ==============================================================================
# 20. SERIALIZE ALL RESPONSE-OBJECTIVE PACKS
# ==============================================================================

print("\n" + "=" * 90)
print("EVIDENCE PACK SERIALIZATION")
print("=" * 90)

group_columns = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
]

grouped = (
    temporal_evidence_df
    .groupby(
        group_columns,
        sort=False,
    )
)

group_count = (
    temporal_evidence_df[
        [
            "response_id",
            "objective_uid",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

assert (
    group_count
    ==
    35_072
), (
    "Expected 35,072 response-objective groups."
)

print(
    "Groups to serialize:",
    f"{group_count:,}",
)


pack_rows = []

groups_processed = 0

for (
    response_id,
    session_id,
    objective_uid,
    fold,
), group_df in grouped:

    groups_processed += 1

    response_id = str(
        response_id
    )

    session_id = str(
        session_id
    )

    objective_uid = str(
        objective_uid
    )

    objective_text = (
        objective_map.get(
            objective_uid
        )
    )

    assert (
        objective_text is not None
    ), (
        "Objective UID missing from catalogue:\n"
        f"{objective_uid}"
    )

    (
        evidence_text,
        actual_tokens,
        sections,
        selected_sections,
    ) = build_pack(
        objective_text,
        group_df,
    )

    # --------------------------------------------------------------------------
    # Source provenance.
    # --------------------------------------------------------------------------

    ordered_group = (
        group_df
        .sort_values(
            [
                "turn_index",
                "turn_uid",
            ],
            kind="mergesort",
        )
    )

    source_turn_uids = (
        ordered_group[
            "turn_uid"
        ]
        .astype(str)
        .tolist()
    )

    source_turn_indices = (
        ordered_group[
            "turn_index"
        ]
        .astype(int)
        .tolist()
    )

    source_roles = (
        ordered_group[
            "canonical_role"
        ]
        .astype(str)
        .tolist()
    )

    source_evidence_types = (
        ordered_group[
            "evidence_source"
        ]
        .astype(str)
        .tolist()
    )

    has_student = bool(
        (
            group_df[
                "canonical_role"
            ]
            ==
            "student"
        ).any()
    )

    has_tutor = bool(
        (
            group_df[
                "canonical_role"
            ]
            ==
            "tutor"
        ).any()
    )

    final_student_present = bool(
        group_df[
            "evidence_source"
        ]
        .astype(str)
        .str.contains(
            "final_student",
            regex=False,
        )
        .any()
    )

    pack_rows.append(
        {
            "response_id": response_id,
            "session_id": session_id,
            "objective_uid": objective_uid,
            "fold": int(fold),

            "objective_text": str(
                objective_text
            ),

            "evidence_text": evidence_text,

            "evidence_token_count": int(
                actual_tokens
            ),

            "selected_sections": json.dumps(
                selected_sections,
                ensure_ascii=False,
            ),

            "source_turn_uids": json.dumps(
                source_turn_uids,
                ensure_ascii=False,
            ),

            "source_turn_indices": json.dumps(
                source_turn_indices,
                ensure_ascii=False,
            ),

            "source_roles": json.dumps(
                source_roles,
                ensure_ascii=False,
            ),

            "source_evidence_types": json.dumps(
                source_evidence_types,
                ensure_ascii=False,
            ),

            "source_turn_count": int(
                len(
                    source_turn_uids
                )
            ),

            "source_min_turn_index": int(
                min(
                    source_turn_indices
                )
            ),

            "source_max_turn_index": int(
                max(
                    source_turn_indices
                )
            ),

            "has_student_evidence": has_student,

            "has_tutor_context": has_tutor,

            "has_final_student_evidence": (
                final_student_present
            ),
        }
    )

    if (
        groups_processed
        %
        5000
        ==
        0
    ):

        print(
            "Groups serialized:",
            f"{groups_processed:,}",
        )

        gc.collect()


del grouped

gc.collect()


# ==============================================================================
# 21. MATERIALIZE PACK DATAFRAME
# ==============================================================================

evidence_packs_df = pd.DataFrame(
    pack_rows
)

del pack_rows

gc.collect()

assert (
    not evidence_packs_df.empty
), (
    "Evidence pack dataframe is empty."
)

print("\n" + "=" * 90)
print("PACK POPULATION")
print("=" * 90)

print(
    "Evidence packs:",
    f"{len(evidence_packs_df):,}",
)


# ==============================================================================
# 22. RESPONSE-OBJECTIVE POPULATION CONTRACT
# ==============================================================================

assert (
    len(evidence_packs_df)
    ==
    35_072
), (
    "Evidence pack population must equal 35,072."
)

assert (
    evidence_packs_df[
        [
            "response_id",
            "objective_uid",
        ]
    ]
    .drop_duplicates()
    .shape[0]
    ==
    35_072
), (
    "Duplicate response-objective packs detected."
)

assert (
    evidence_packs_df[
        "response_id"
    ].is_unique
), (
    "response_id must be unique."
)

print(
    "One row = one response-objective : PASS"
)


# ==============================================================================
# 23. JOIN CANONICAL TARGET
# ==============================================================================

target_join = (
    response_meta[
        [
            "response_id",
            TARGET_COLUMN,
        ]
    ]
    .copy()
)

target_join[
    "response_id"
] = (
    target_join[
        "response_id"
    ]
    .astype(str)
)

evidence_packs_df = (
    evidence_packs_df
    .merge(
        target_join,
        on="response_id",
        how="left",
        validate="one_to_one",
    )
)

assert (
    evidence_packs_df[
        TARGET_COLUMN
    ]
    .notna()
    .all()
), (
    "Missing target after canonical response join."
)

evidence_packs_df[
    TARGET_COLUMN
] = (
    evidence_packs_df[
        TARGET_COLUMN
    ]
    .astype(int)
)

# Standardize downstream training name.
evidence_packs_df[
    "target"
] = (
    evidence_packs_df[
        TARGET_COLUMN
    ]
    .astype(int)
)

print(
    "Canonical target join : PASS"
)


# ==============================================================================
# 24. TOKEN BUDGET CONTRACT
# ==============================================================================

token_counts = (
    evidence_packs_df[
        "evidence_token_count"
    ]
    .astype(int)
)

assert (
    token_counts
    >=
    1
).all(), (
    "Evidence pack contains zero tokens."
)

assert (
    token_counts
    <=
    MAX_EVIDENCE_TOKENS
).all(), (
    "Evidence pack exceeds 2048 tokens."
)

print("\n" + "=" * 90)
print("TOKEN BUDGET")
print("=" * 90)

print(
    "Maximum:",
    int(
        token_counts.max()
    ),
)

print(
    "Mean:",
    round(
        float(
            token_counts.mean()
        ),
        2,
    ),
)

print(
    "Median:",
    float(
        token_counts.median()
    ),
)

print(
    "P95:",
    float(
        token_counts.quantile(
            0.95
        )
    ),
)

print(
    "P99:",
    float(
        token_counts.quantile(
            0.99
        )
    ),
)

print(
    "2048-token contract : PASS"
)


# ==============================================================================
# 25. FINAL TEXT INTEGRITY
# ==============================================================================

assert (
    evidence_packs_df[
        "evidence_text"
    ]
    .notna()
    .all()
), (
    "evidence_text contains nulls."
)

assert (
    evidence_packs_df[
        "evidence_text"
    ]
    .astype(str)
    .str.strip()
    .ne("")
    .all()
), (
    "evidence_text contains empty strings."
)

assert (
    evidence_packs_df[
        "objective_text"
    ]
    .astype(str)
    .str.strip()
    .ne("")
    .all()
), (
    "objective_text contains empty strings."
)

print(
    "Evidence text integrity : PASS"
)


# ==============================================================================
# 26. COVERAGE DIAGNOSTICS
# ==============================================================================

student_coverage = float(
    evidence_packs_df[
        "has_student_evidence"
    ].mean()
)

tutor_coverage = float(
    evidence_packs_df[
        "has_tutor_context"
    ].mean()
)

final_student_coverage = float(
    evidence_packs_df[
        "has_final_student_evidence"
    ].mean()
)

print("\n" + "=" * 90)
print("PACK COVERAGE")
print("=" * 90)

print(
    "Student evidence coverage:",
    f"{student_coverage:.6f}",
)

print(
    "Tutor context coverage:",
    f"{tutor_coverage:.6f}",
)

print(
    "Final student evidence coverage:",
    f"{final_student_coverage:.6f}",
)


# ==============================================================================
# 27. FOLD DISTRIBUTION
# ==============================================================================

fold_distribution = (
    evidence_packs_df[
        "fold"
    ]
    .value_counts(
        sort=True
    )
    .sort_index()
)

print("\n" + "=" * 90)
print("FOLD DISTRIBUTION")
print("=" * 90)

for fold_value, count in (
    fold_distribution.items()
):

    print(
        f"Fold {int(fold_value)}:",
        f"{int(count):,}",
    )

assert (
    set(
        fold_distribution.index.astype(int)
    )
    ==
    {0, 1, 2, 3, 4}
), (
    "Expected folds 0-4."
)


# ==============================================================================
# 28. TARGET DISTRIBUTION
# ==============================================================================

target_distribution = (
    evidence_packs_df[
        "target"
    ]
    .value_counts(
        sort=True
    )
    .sort_index()
)

print("\n" + "=" * 90)
print("TARGET DISTRIBUTION")
print("=" * 90)

for target_value, count in (
    target_distribution.items()
):

    print(
        f"Target {int(target_value)}:",
        f"{int(count):,}",
    )


# ==============================================================================
# 29. SAVE FINAL EVIDENCE PACK ARTIFACT
# ==============================================================================

EVIDENCE_PACK_PATH = (
    EVIDENCE_PACK_ROOT
    / "evidence_packs.parquet"
)

evidence_packs_df.to_parquet(
    EVIDENCE_PACK_PATH,
    index=False,
    compression="zstd",
)

assert (
    EVIDENCE_PACK_PATH.exists()
), (
    "Evidence pack parquet was not created."
)

print("\n" + "=" * 90)
print("EVIDENCE PACK ARTIFACT")
print("=" * 90)

print(
    "Path:",
    EVIDENCE_PACK_PATH,
)

print(
    "Rows:",
    f"{len(evidence_packs_df):,}",
)

print(
    "Size:",
    f"{EVIDENCE_PACK_PATH.stat().st_size:,} bytes",
)


# ==============================================================================
# 30. SERIALIZATION DIAGNOSTICS
# ==============================================================================

SERIALIZATION_DIAGNOSTICS_PATH = (
    SERIALIZATION_AUDIT_ROOT
    / "evidence_pack_serialization_diagnostics.parquet"
)

serialization_diagnostics = (
    evidence_packs_df[
        [
            "response_id",
            "session_id",
            "objective_uid",
            "fold",
            "evidence_token_count",
            "source_turn_count",
            "source_min_turn_index",
            "source_max_turn_index",
            "has_student_evidence",
            "has_tutor_context",
            "has_final_student_evidence",
        ]
    ]
    .copy()
)

serialization_diagnostics.to_parquet(
    SERIALIZATION_DIAGNOSTICS_PATH,
    index=False,
    compression="zstd",
)


# ==============================================================================
# 31. TOKEN DISTRIBUTION AUDIT
# ==============================================================================

TOKEN_DISTRIBUTION_PATH = (
    SERIALIZATION_AUDIT_ROOT
    / "evidence_token_distribution.parquet"
)

token_distribution = pd.DataFrame(
    {
        "metric": [
            "min",
            "max",
            "mean",
            "median",
            "p90",
            "p95",
            "p99",
        ],
        "value": [
            float(
                token_counts.min()
            ),
            float(
                token_counts.max()
            ),
            float(
                token_counts.mean()
            ),
            float(
                token_counts.median()
            ),
            float(
                token_counts.quantile(
                    0.90
                )
            ),
            float(
                token_counts.quantile(
                    0.95
                )
            ),
            float(
                token_counts.quantile(
                    0.99
                )
            ),
        ],
    }
)

token_distribution.to_parquet(
    TOKEN_DISTRIBUTION_PATH,
    index=False,
    compression="zstd",
)


# ==============================================================================
# 32. SAMPLE ARTIFACT
# ==============================================================================

SAMPLE_PATH = (
    SERIALIZATION_AUDIT_ROOT
    / "evidence_pack_sample.parquet"
)

sample_columns = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "objective_text",
    "evidence_text",
    "evidence_token_count",
    "target",
]

sample_df = (
    evidence_packs_df[
        sample_columns
    ]
    .head(100)
    .copy()
)

sample_df.to_parquet(
    SAMPLE_PATH,
    index=False,
    compression="zstd",
)


# ==============================================================================
# 33. CELL 5 GATE
# ==============================================================================

EVIDENCE_PACK_CELL_5_READY = True

print("\n" + "=" * 90)
print(
    "EVIDENCE PACK CELL 5 — "
    "2048-TOKEN SERIALIZATION: PASS"
)
print("=" * 90)

print(
    "Evidence packs:",
    EVIDENCE_PACK_PATH,
)

print(
    "Serialization diagnostics:",
    SERIALIZATION_DIAGNOSTICS_PATH,
)

print(
    "Token distribution:",
    TOKEN_DISTRIBUTION_PATH,
)

print(
    "Sample:",
    SAMPLE_PATH,
)


# ==============================================================================
# 34. MEMORY CLEANUP
# ==============================================================================

for _name in [
    "temporal_evidence_df",
    "objective_catalogue",
    "objective_map",
    "response_meta",
    "target_join",
    "response_pf",
    "tokenizer",
    "evidence_packs_df",
    "serialization_diagnostics",
    "token_distribution",
    "sample_df",
    "token_counts",
    "fold_distribution",
    "target_distribution",
]:

    if _name in globals():
        del globals()[_name]


gc.collect()

print(
    "Cell 5 memory cleanup: PASS"
)

TRACE THE ACE — EVIDENCE PACK BUILDER
CELL 5 — 2048-TOKEN EVIDENCE PACK SERIALIZATION

DEPENDENCY GATE
Cell 4 dependency : PASS

INPUT ARTIFACTS
Temporal evidence: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\03_evidence_pack\audit\temporal\temporal_evidence_candidates.parquet
Objective catalogue: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input\objective_catalogue.parquet
R0 session-turn index: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input\session_turn_index.parquet

TEMPORAL EVIDENCE INPUT
Rows: 1,063,637
Columns: ['response_id', 'session_id', 'objective_uid', 'fold', 'turn_uid', 'turn_index', 'role', 'canonical_role', 'text_norm', 'evidence_source', 'cross_encoder_rank', 'cross_encoder_score', 'selection_priority']
Temporal evidence schema : PASS

OBJECTIVE CATALOGUE
Rows: 398
Objective text column: objective_raw
Objective map : PASS

CANONICAL RESPONSE ARTIFACT
Response artifact: D:\Competition\Trac

In [18]:
# ==============================================================================
# TRACE THE ACE — EVIDENCE PACK BUILDER
# CELL 6 — FINAL EVIDENCE PACK FREEZE + INTEGRITY MANIFEST
# ==============================================================================

import gc
import hashlib
import json
import os
import shutil
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


print("=" * 90)
print("TRACE THE ACE — EVIDENCE PACK BUILDER")
print("CELL 6 — FINAL EVIDENCE PACK FREEZE + INTEGRITY MANIFEST")
print("=" * 90)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    "EVIDENCE_PACK_CELL_5_READY" in globals()
    and EVIDENCE_PACK_CELL_5_READY is True
), (
    "Cell 5 dependency failed."
)

print("\n" + "=" * 90)
print("DEPENDENCY GATE")
print("=" * 90)

print(
    "Cell 5 dependency : PASS"
)


# ==============================================================================
# 2. PROJECT PATHS
# ==============================================================================

PROJECT_ROOT = Path(
    PROJECT_ROOT
)

SCRATCH_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
)

EVIDENCE_ROOT = (
    SCRATCH_ROOT
    / "03_evidence_pack"
)

EVIDENCE_PACK_ROOT = (
    EVIDENCE_ROOT
    / "packs"
)

EVIDENCE_FREEZE_ROOT = (
    EVIDENCE_ROOT
    / "frozen"
)

EVIDENCE_FREEZE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

EVIDENCE_PACK_INPUT_PATH = (
    EVIDENCE_PACK_ROOT
    / "evidence_packs.parquet"
)

FROZEN_EVIDENCE_PATH = (
    EVIDENCE_FREEZE_ROOT
    / "evidence_packs.parquet"
)

FREEZE_MANIFEST_PATH = (
    EVIDENCE_FREEZE_ROOT
    / "cell6_freeze_manifest.json"
)

FREEZE_TMP_PATH = (
    EVIDENCE_FREEZE_ROOT
    / "evidence_packs.freeze.tmp.parquet"
)


print("\n" + "=" * 90)
print("FREEZE PATHS")
print("=" * 90)

print(
    "Input:",
    EVIDENCE_PACK_INPUT_PATH,
)

print(
    "Freeze root:",
    EVIDENCE_FREEZE_ROOT,
)

print(
    "Frozen candidates:",
    FROZEN_EVIDENCE_PATH,
)

print(
    "Freeze manifest:",
    FREEZE_MANIFEST_PATH,
)


# ==============================================================================
# 3. INPUT ARTIFACT EXISTENCE
# ==============================================================================

assert (
    EVIDENCE_PACK_INPUT_PATH.exists()
), (
    "Cell 5 evidence pack artifact is missing:\n"
    f"{EVIDENCE_PACK_INPUT_PATH}"
)

assert (
    EVIDENCE_PACK_INPUT_PATH.is_file()
), (
    "Evidence pack input is not a file."
)

print("\n" + "=" * 90)
print("INPUT ARTIFACT")
print("=" * 90)

print(
    "Evidence pack parquet: PASS"
)


# ==============================================================================
# 4. LOAD PARQUET METADATA ONLY
# ==============================================================================

input_pf = pq.ParquetFile(
    EVIDENCE_PACK_INPUT_PATH
)

input_schema = (
    input_pf.schema_arrow
)

input_columns = (
    input_schema.names
)

input_rows = (
    input_pf.metadata.num_rows
)

input_row_groups = (
    input_pf.metadata.num_row_groups
)

print("\n" + "=" * 90)
print("INPUT PARQUET METADATA")
print("=" * 90)

print(
    "Rows:",
    f"{input_rows:,}",
)

print(
    "Row groups:",
    f"{input_row_groups:,}",
)

print(
    "Columns:",
    input_columns,
)


# ==============================================================================
# 5. POPULATION CONTRACT
# ==============================================================================

EXPECTED_PACK_ROWS = 35_072

assert (
    input_rows
    ==
    EXPECTED_PACK_ROWS
), (
    "Evidence pack population mismatch.\n"
    f"Expected: {EXPECTED_PACK_ROWS:,}\n"
    f"Observed: {input_rows:,}"
)

print(
    "Population contract : PASS"
)


# ==============================================================================
# 6. SCHEMA CONTRACT
# ==============================================================================

REQUIRED_FROZEN_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "objective_text",
    "evidence_text",
    "evidence_token_count",
    "selected_sections",
    "source_turn_uids",
    "source_turn_indices",
    "source_roles",
    "source_evidence_types",
    "source_turn_count",
    "source_min_turn_index",
    "source_max_turn_index",
    "has_student_evidence",
    "has_tutor_context",
    "has_final_student_evidence",
    "target",
]

missing_frozen_columns = sorted(
    set(REQUIRED_FROZEN_COLUMNS)
    -
    set(input_columns)
)

assert not missing_frozen_columns, (
    "Evidence pack is missing required frozen columns:\n"
    f"{missing_frozen_columns}"
)

print(
    "Schema contract : PASS"
)


# ==============================================================================
# 7. STREAMED CONTENT AUDIT
# ==============================================================================

print("\n" + "=" * 90)
print("STREAMED CONTENT AUDIT")
print("=" * 90)

observed_rows = 0

response_ids = set()
objective_keys = set()

fold_counts = {}
target_counts = {}

min_tokens = None
max_tokens = None

invalid_token_rows = 0
empty_evidence_rows = 0
empty_objective_rows = 0

duplicate_response_ids = 0
duplicate_response_objective_keys = 0

seen_response_ids = set()
seen_response_objective_keys = set()


scanner = input_pf.iter_batches(
    batch_size=100_000
)

for batch_index, batch in enumerate(
    scanner,
    start=1,
):

    batch_df = (
        batch
        .to_pandas()
    )

    observed_rows += len(
        batch_df
    )

    # --------------------------------------------------------------------------
    # Identity
    # --------------------------------------------------------------------------

    batch_response_ids = (
        batch_df[
            "response_id"
        ]
        .astype(str)
    )

    batch_objective_uids = (
        batch_df[
            "objective_uid"
        ]
        .astype(str)
    )

    for response_id in (
        batch_response_ids
    ):

        if response_id in seen_response_ids:

            duplicate_response_ids += 1

        seen_response_ids.add(
            response_id
        )

    for response_id, objective_uid in zip(
        batch_response_ids,
        batch_objective_uids,
    ):

        key = (
            response_id,
            objective_uid,
        )

        if key in seen_response_objective_keys:

            duplicate_response_objective_keys += 1

        seen_response_objective_keys.add(
            key
        )

    # --------------------------------------------------------------------------
    # Token contract
    # --------------------------------------------------------------------------

    token_values = pd.to_numeric(
        batch_df[
            "evidence_token_count"
        ],
        errors="coerce",
    )

    invalid_token_rows += int(
        (
            token_values.isna()
            |
            ~np.isfinite(
                token_values.to_numpy(
                    dtype=float
                )
            )
            |
            (token_values < 1)
            |
            (token_values > 2048)
        ).sum()
    )

    if len(token_values):

        batch_min = int(
            token_values.min()
        )

        batch_max = int(
            token_values.max()
        )

        if min_tokens is None:

            min_tokens = batch_min

        else:

            min_tokens = min(
                min_tokens,
                batch_min,
            )

        if max_tokens is None:

            max_tokens = batch_max

        else:

            max_tokens = max(
                max_tokens,
                batch_max,
            )

    # --------------------------------------------------------------------------
    # Text contract
    # --------------------------------------------------------------------------

    empty_evidence_rows += int(
        (
            batch_df[
                "evidence_text"
            ]
            .isna()
            |
            batch_df[
                "evidence_text"
            ]
            .astype(str)
            .str.strip()
            .eq("")
        ).sum()
    )

    empty_objective_rows += int(
        (
            batch_df[
                "objective_text"
            ]
            .isna()
            |
            batch_df[
                "objective_text"
            ]
            .astype(str)
            .str.strip()
            .eq("")
        ).sum()
    )

    # --------------------------------------------------------------------------
    # Fold / target distribution
    # --------------------------------------------------------------------------

    for value, count in (
        batch_df[
            "fold"
        ]
        .value_counts()
        .items()
    ):

        value = int(value)

        fold_counts[value] = (
            fold_counts.get(
                value,
                0,
            )
            +
            int(count)
        )

    for value, count in (
        batch_df[
            "target"
        ]
        .value_counts()
        .items()
    ):

        value = int(value)

        target_counts[value] = (
            target_counts.get(
                value,
                0,
            )
            +
            int(count)
        )

    if (
        batch_index
        %
        5
        ==
        0
    ):

        print(
            "Processed rows:",
            f"{observed_rows:,}",
        )

    del batch_df
    del token_values

    gc.collect()


del scanner

gc.collect()


# ==============================================================================
# 8. STREAMED AUDIT ASSERTIONS
# ==============================================================================

assert (
    observed_rows
    ==
    input_rows
), (
    "Streamed row count differs from parquet metadata."
)

assert (
    observed_rows
    ==
    EXPECTED_PACK_ROWS
), (
    "Streamed evidence pack population mismatch."
)

assert (
    duplicate_response_ids
    ==
    0
), (
    "Duplicate response_id detected."
)

assert (
    duplicate_response_objective_keys
    ==
    0
), (
    "Duplicate response-objective key detected."
)

assert (
    invalid_token_rows
    ==
    0
), (
    "Invalid evidence_token_count detected."
)

assert (
    empty_evidence_rows
    ==
    0
), (
    "Empty evidence_text rows detected."
)

assert (
    empty_objective_rows
    ==
    0
), (
    "Empty objective_text rows detected."
)

assert (
    min_tokens
    is not None
), (
    "Could not determine minimum token count."
)

assert (
    max_tokens
    is not None
), (
    "Could not determine maximum token count."
)

assert (
    max_tokens
    <=
    2048
), (
    "Frozen evidence exceeds 2048 tokens."
)

assert (
    set(
        fold_counts.keys()
    )
    ==
    {0, 1, 2, 3, 4}
), (
    "Frozen evidence does not contain folds 0-4."
)

print(
    "Streamed row count : PASS"
)

print(
    "Duplicate response IDs : PASS"
)

print(
    "Duplicate response-objective keys : PASS"
)

print(
    "Text integrity : PASS"
)

print(
    "Token integrity : PASS"
)

print(
    "Token range:",
    f"{min_tokens} - {max_tokens}",
)

print(
    "Fold coverage : PASS"
)


# ==============================================================================
# 9. STREAMING SHA-256
# ==============================================================================

def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


print("\n" + "=" * 90)
print("INPUT SHA-256")
print("=" * 90)

INPUT_SHA256 = sha256_file(
    EVIDENCE_PACK_INPUT_PATH
)

print(
    "SHA-256:",
    INPUT_SHA256,
)


# ==============================================================================
# 10. REMOVE STALE TEMPORARY FREEZE
# ==============================================================================

if FREEZE_TMP_PATH.exists():

    FREEZE_TMP_PATH.unlink()

    print(
        "Removed stale temporary freeze : PASS"
    )


# ==============================================================================
# 11. COPY FREEZE — STREAMING FILE COPY
# ==============================================================================

print("\n" + "=" * 90)
print("FREEZE COPY")
print("=" * 90)

with open(
    EVIDENCE_PACK_INPUT_PATH,
    "rb",
) as source_handle:

    with open(
        FREEZE_TMP_PATH,
        "wb",
    ) as target_handle:

        while True:

            chunk = source_handle.read(
                16 * 1024 * 1024
            )

            if not chunk:
                break

            target_handle.write(
                chunk
            )


assert (
    FREEZE_TMP_PATH.exists()
), (
    "Temporary frozen artifact was not created."
)

assert (
    FREEZE_TMP_PATH.stat().st_size
    ==
    EVIDENCE_PACK_INPUT_PATH.stat().st_size
), (
    "Temporary freeze size differs from input."
)


# ==============================================================================
# 12. FREEZE COPY SHA-256
# ==============================================================================

FROZEN_TMP_SHA256 = sha256_file(
    FREEZE_TMP_PATH
)

assert (
    FROZEN_TMP_SHA256
    ==
    INPUT_SHA256
), (
    "Frozen temporary artifact SHA-256 mismatch."
)

print(
    "Temporary freeze SHA-256 : PASS"
)


# ==============================================================================
# 13. ATOMIC FINAL REPLACEMENT
# ==============================================================================

if FROZEN_EVIDENCE_PATH.exists():

    FROZEN_EVIDENCE_PATH.unlink()

FREEZE_TMP_PATH.replace(
    FROZEN_EVIDENCE_PATH
)

assert (
    FROZEN_EVIDENCE_PATH.exists()
), (
    "Final frozen evidence artifact missing."
)

print(
    "Atomic freeze replacement : PASS"
)


# ==============================================================================
# 14. FINAL FROZEN HASH
# ==============================================================================

FROZEN_SHA256 = sha256_file(
    FROZEN_EVIDENCE_PATH
)

assert (
    FROZEN_SHA256
    ==
    INPUT_SHA256
), (
    "Final frozen artifact SHA-256 mismatch."
)

print(
    "Final frozen SHA-256 : PASS"
)


# ==============================================================================
# 15. FINAL FROZEN PARQUET METADATA
# ==============================================================================

frozen_pf = pq.ParquetFile(
    FROZEN_EVIDENCE_PATH
)

frozen_rows = (
    frozen_pf.metadata.num_rows
)

frozen_row_groups = (
    frozen_pf.metadata.num_row_groups
)

frozen_schema = (
    frozen_pf.schema_arrow
)

frozen_columns = (
    frozen_schema.names
)

assert (
    frozen_rows
    ==
    EXPECTED_PACK_ROWS
), (
    "Frozen parquet row count mismatch."
)

assert (
    frozen_columns
    ==
    input_columns
), (
    "Frozen schema differs from input schema."
)

print("\n" + "=" * 90)
print("FROZEN ARTIFACT")
print("=" * 90)

print(
    "Frozen rows:",
    f"{frozen_rows:,}",
)

print(
    "Frozen row groups:",
    f"{frozen_row_groups:,}",
)

print(
    "Frozen schema : PASS"
)


# ==============================================================================
# 16. SCHEMA FINGERPRINT
# ==============================================================================

schema_payload = (
    str(
        frozen_schema
    )
    .encode(
        "utf-8"
    )
)

SCHEMA_SHA256 = hashlib.sha256(
    schema_payload
).hexdigest()

print(
    "Schema SHA-256:",
    SCHEMA_SHA256,
)


# ==============================================================================
# 17. FROZEN ARTIFACT SIZE
# ==============================================================================

FROZEN_SIZE_BYTES = (
    FROZEN_EVIDENCE_PATH.stat().st_size
)

assert (
    FROZEN_SIZE_BYTES
    >
    0
), (
    "Frozen evidence parquet is empty."
)

print(
    "Frozen size:",
    f"{FROZEN_SIZE_BYTES:,} bytes",
)


# ==============================================================================
# 18. MANIFEST
# ==============================================================================

freeze_timestamp = (
    datetime.now(
        timezone.utc
    )
    .isoformat()
)

manifest = {
    "status": "FROZEN",

    "artifact": "evidence_packs",

    "stage": "03_evidence_pack",

    "cell": 6,

    "created_at_utc": (
        freeze_timestamp
    ),

    "project_root": str(
        PROJECT_ROOT
    ),

    "input_artifact": str(
        EVIDENCE_PACK_INPUT_PATH
    ),

    "frozen_artifact": str(
        FROZEN_EVIDENCE_PATH
    ),

    "rows": int(
        frozen_rows
    ),

    "row_groups": int(
        frozen_row_groups
    ),

    "size_bytes": int(
        FROZEN_SIZE_BYTES
    ),

    "sha256": FROZEN_SHA256,

    "candidate_sha256": FROZEN_SHA256,

    "input_sha256": INPUT_SHA256,

    "schema_sha256": SCHEMA_SHA256,

    "columns": frozen_columns,

    "expected_rows": int(
        EXPECTED_PACK_ROWS
    ),

    "max_evidence_tokens": 2048,

    "observed_min_evidence_tokens": int(
        min_tokens
    ),

    "observed_max_evidence_tokens": int(
        max_tokens
    ),

    "fold_counts": {
        str(
            key
        ): int(
            value
        )
        for key, value in sorted(
            fold_counts.items()
        )
    },

    "target_counts": {
        str(
            key
        ): int(
            value
        )
        for key, value in sorted(
            target_counts.items()
        )
    },

    "integrity": {
        "population_contract": True,
        "schema_contract": True,
        "duplicate_response_id_check": True,
        "duplicate_response_objective_check": True,
        "text_integrity": True,
        "token_budget_contract": True,
        "fold_coverage": True,
        "sha256_verified": True,
        "frozen_reload_metadata_verified": True,
    },

    "upstream": {
        "cell_5_status": "PASS",
    },
}


# ==============================================================================
# 19. WRITE MANIFEST
# ==============================================================================

with open(
    FREEZE_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )


assert (
    FREEZE_MANIFEST_PATH.exists()
), (
    "Freeze manifest was not created."
)

print("\n" + "=" * 90)
print("FREEZE MANIFEST")
print("=" * 90)

print(
    "Manifest:",
    FREEZE_MANIFEST_PATH,
)

print(
    "Status:",
    manifest[
        "status"
    ],
)

print(
    "Artifact:",
    manifest[
        "artifact"
    ],
)


# ==============================================================================
# 20. MANIFEST RELOAD / SELF-CHECK
# ==============================================================================

with open(
    FREEZE_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as handle:

    manifest_reload = json.load(
        handle
    )

assert (
    manifest_reload[
        "status"
    ]
    ==
    "FROZEN"
), (
    "Freeze manifest status is not FROZEN."
)

assert (
    manifest_reload[
        "rows"
    ]
    ==
    EXPECTED_PACK_ROWS
), (
    "Manifest row count mismatch."
)

assert (
    manifest_reload[
        "sha256"
    ]
    ==
    FROZEN_SHA256
), (
    "Manifest SHA-256 mismatch."
)

assert (
    manifest_reload[
        "schema_sha256"
    ]
    ==
    SCHEMA_SHA256
), (
    "Manifest schema SHA-256 mismatch."
)

assert (
    manifest_reload[
        "max_evidence_tokens"
    ]
    ==
    2048
), (
    "Manifest token budget mismatch."
)

print(
    "Manifest JSON reload : PASS"
)


# ==============================================================================
# 21. FINAL CELL 6 CONTRACT
# ==============================================================================

assert (
    FROZEN_EVIDENCE_PATH.exists()
), (
    "Final frozen evidence artifact does not exist."
)

assert (
    FREEZE_MANIFEST_PATH.exists()
), (
    "Final freeze manifest does not exist."
)

assert (
    FROZEN_SHA256
    ==
    INPUT_SHA256
), (
    "Input/frozen SHA-256 mismatch."
)

assert (
    frozen_rows
    ==
    EXPECTED_PACK_ROWS
), (
    "Final frozen population mismatch."
)

assert (
    max_tokens
    <=
    2048
), (
    "Final frozen token budget violation."
)


# ==============================================================================
# 22. CELL 6 READY FLAG
# ==============================================================================

EVIDENCE_PACK_CELL_6_READY = True

print("\n" + "=" * 90)
print(
    "EVIDENCE PACK CELL 6 — "
    "FINAL FREEZE + INTEGRITY MANIFEST: PASS"
)
print("=" * 90)

print(
    "Frozen evidence:",
    FROZEN_EVIDENCE_PATH,
)

print(
    "Freeze manifest:",
    FREEZE_MANIFEST_PATH,
)

print(
    "Rows:",
    f"{frozen_rows:,}",
)

print(
    "SHA-256:",
    FROZEN_SHA256,
)

print(
    "Token range:",
    f"{min_tokens} - {max_tokens}",
)


# ==============================================================================
# 23. MEMORY CLEANUP
# ==============================================================================

for _name in [
    "input_pf",
    "input_schema",
    "frozen_pf",
    "frozen_schema",
    "manifest",
    "manifest_reload",
    "response_columns",
    "response_pf",
]:

    if _name in globals():
        del globals()[_name]


gc.collect()

print(
    "Cell 6 memory cleanup: PASS"
)

TRACE THE ACE — EVIDENCE PACK BUILDER
CELL 6 — FINAL EVIDENCE PACK FREEZE + INTEGRITY MANIFEST

DEPENDENCY GATE
Cell 5 dependency : PASS

FREEZE PATHS
Input: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\03_evidence_pack\packs\evidence_packs.parquet
Freeze root: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\03_evidence_pack\frozen
Frozen candidates: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\03_evidence_pack\frozen\evidence_packs.parquet
Freeze manifest: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\03_evidence_pack\frozen\cell6_freeze_manifest.json

INPUT ARTIFACT
Evidence pack parquet: PASS

INPUT PARQUET METADATA
Rows: 35,072
Row groups: 1
Columns: ['response_id', 'session_id', 'objective_uid', 'fold', 'objective_text', 'evidence_text', 'evidence_token_count', 'selected_sections', 'source_turn_uids', 'source_turn_indices', 'source_roles', 'source_evidence_types', 'source_turn_count', 'source_min_turn_index', 'source_max_turn

In [19]:
# ==============================================================================
# TRACE THE ACE — EVIDENCE PACK BUILDER
# CELL 7 — FROZEN RELOAD + EXACT INTEGRITY VERIFICATION
# ==============================================================================

import gc
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


print("=" * 90)
print("TRACE THE ACE — EVIDENCE PACK BUILDER")
print("CELL 7 — FROZEN RELOAD + EXACT INTEGRITY VERIFICATION")
print("=" * 90)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    "EVIDENCE_PACK_CELL_6_READY" in globals()
    and EVIDENCE_PACK_CELL_6_READY is True
), (
    "Cell 6 dependency failed."
)

print("\n" + "=" * 90)
print("DEPENDENCY GATE")
print("=" * 90)

print(
    "Cell 6 dependency : PASS"
)


# ==============================================================================
# 2. PATHS
# ==============================================================================

PROJECT_ROOT = Path(
    PROJECT_ROOT
)

SCRATCH_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
)

EVIDENCE_ROOT = (
    SCRATCH_ROOT
    / "03_evidence_pack"
)

EVIDENCE_FREEZE_ROOT = (
    EVIDENCE_ROOT
    / "frozen"
)

FROZEN_EVIDENCE_PATH = (
    EVIDENCE_FREEZE_ROOT
    / "evidence_packs.parquet"
)

FREEZE_MANIFEST_PATH = (
    EVIDENCE_FREEZE_ROOT
    / "cell6_freeze_manifest.json"
)


print("\n" + "=" * 90)
print("FROZEN ARTIFACT PATHS")
print("=" * 90)

print(
    "Frozen root :",
    EVIDENCE_FREEZE_ROOT,
)

print(
    "Candidates  :",
    FROZEN_EVIDENCE_PATH,
)

print(
    "Manifest    :",
    FREEZE_MANIFEST_PATH,
)


# ==============================================================================
# 3. ARTIFACT EXISTENCE
# ==============================================================================

assert (
    EVIDENCE_FREEZE_ROOT.exists()
), (
    "Evidence freeze root is missing."
)

assert (
    FROZEN_EVIDENCE_PATH.exists()
), (
    "Frozen evidence parquet is missing:\n"
    f"{FROZEN_EVIDENCE_PATH}"
)

assert (
    FREEZE_MANIFEST_PATH.exists()
), (
    "Frozen evidence manifest is missing:\n"
    f"{FREEZE_MANIFEST_PATH}"
)

print("\n" + "=" * 90)
print("ARTIFACT EXISTENCE")
print("=" * 90)

print(
    "Frozen parquet : PASS"
)

print(
    "Freeze manifest: PASS"
)


# ==============================================================================
# 4. MANIFEST RELOAD
# ==============================================================================

with open(
    FREEZE_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as handle:

    R7_MANIFEST = json.load(
        handle
    )

assert isinstance(
    R7_MANIFEST,
    dict,
), (
    "Freeze manifest did not deserialize to a JSON object."
)

print("\n" + "=" * 90)
print("FREEZE MANIFEST")
print("=" * 90)

print(
    "JSON valid : PASS"
)

print(
    "Status:",
    R7_MANIFEST.get(
        "status"
    ),
)

print(
    "Artifact:",
    R7_MANIFEST.get(
        "artifact"
    ),
)


assert (
    R7_MANIFEST.get(
        "status"
    )
    ==
    "FROZEN"
), (
    "Freeze manifest status is not FROZEN."
)

assert (
    R7_MANIFEST.get(
        "artifact"
    )
    ==
    "evidence_packs"
), (
    "Unexpected frozen artifact name."
)


# ==============================================================================
# 5. PARQUET METADATA RELOAD
# ==============================================================================

frozen_pf = pq.ParquetFile(
    FROZEN_EVIDENCE_PATH
)

frozen_schema = (
    frozen_pf.schema_arrow
)

frozen_columns = (
    frozen_schema.names
)

frozen_rows = (
    frozen_pf.metadata.num_rows
)

frozen_row_groups = (
    frozen_pf.metadata.num_row_groups
)


print("\n" + "=" * 90)
print("FROZEN PARQUET METADATA")
print("=" * 90)

print(
    "Rows:",
    f"{frozen_rows:,}",
)

print(
    "Row groups:",
    f"{frozen_row_groups:,}",
)

print(
    "Columns:",
    frozen_columns,
)


# ==============================================================================
# 6. POPULATION CONTRACT
# ==============================================================================

EXPECTED_PACK_ROWS = 35_072

manifest_rows = int(
    R7_MANIFEST.get(
        "rows",
        -1,
    )
)

assert (
    frozen_rows
    ==
    EXPECTED_PACK_ROWS
), (
    "Frozen population mismatch.\n"
    f"Expected: {EXPECTED_PACK_ROWS:,}\n"
    f"Observed: {frozen_rows:,}"
)

assert (
    frozen_rows
    ==
    manifest_rows
), (
    "Frozen parquet row count differs from manifest."
)

print(
    "Population contract : PASS"
)


# ==============================================================================
# 7. SCHEMA CONTRACT
# ==============================================================================

REQUIRED_FROZEN_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "objective_text",
    "evidence_text",
    "evidence_token_count",
    "selected_sections",
    "source_turn_uids",
    "source_turn_indices",
    "source_roles",
    "source_evidence_types",
    "source_turn_count",
    "source_min_turn_index",
    "source_max_turn_index",
    "has_student_evidence",
    "has_tutor_context",
    "has_final_student_evidence",
    "target",
]

missing_columns = sorted(
    set(REQUIRED_FROZEN_COLUMNS)
    -
    set(frozen_columns)
)

assert not missing_columns, (
    "Frozen evidence is missing required columns:\n"
    f"{missing_columns}"
)

manifest_columns = (
    R7_MANIFEST.get(
        "columns"
    )
)

assert (
    manifest_columns
    ==
    frozen_columns
), (
    "Frozen schema differs from manifest schema."
)

print(
    "Schema contract : PASS"
)


# ==============================================================================
# 8. SCHEMA FINGERPRINT
# ==============================================================================

schema_sha256_observed = hashlib.sha256(
    str(
        frozen_schema
    ).encode(
        "utf-8"
    )
).hexdigest()

schema_sha256_expected = (
    R7_MANIFEST.get(
        "schema_sha256"
    )
)

assert (
    schema_sha256_expected
), (
    "Manifest does not contain schema_sha256."
)

assert (
    schema_sha256_observed
    ==
    schema_sha256_expected
), (
    "Schema SHA-256 mismatch.\n"
    f"Expected: {schema_sha256_expected}\n"
    f"Observed: {schema_sha256_observed}"
)

print(
    "Schema fingerprint : PASS"
)


# ==============================================================================
# 9. STREAMING SHA-256
# ==============================================================================

def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


print("\n" + "=" * 90)
print("FROZEN SHA-256")
print("=" * 90)

sha256_observed = sha256_file(
    FROZEN_EVIDENCE_PATH
)

sha256_expected = (
    R7_MANIFEST.get(
        "sha256"
    )
)

assert (
    sha256_expected
), (
    "Manifest does not contain sha256."
)

assert (
    sha256_observed
    ==
    sha256_expected
), (
    "Frozen evidence SHA-256 mismatch.\n"
    f"Expected: {sha256_expected}\n"
    f"Observed: {sha256_observed}"
)

print(
    "SHA-256:",
    sha256_observed,
)

print(
    "SHA-256 integrity : PASS"
)


# ==============================================================================
# 10. FILE SIZE CONTRACT
# ==============================================================================

observed_size_bytes = (
    FROZEN_EVIDENCE_PATH.stat()
    .st_size
)

manifest_size_bytes = int(
    R7_MANIFEST.get(
        "size_bytes",
        -1,
    )
)

assert (
    observed_size_bytes
    ==
    manifest_size_bytes
), (
    "Frozen artifact size differs from manifest."
)

assert (
    observed_size_bytes
    >
    0
), (
    "Frozen artifact is empty."
)

print("\n" + "=" * 90)
print("FILE SIZE")
print("=" * 90)

print(
    "Size:",
    f"{observed_size_bytes:,} bytes",
)

print(
    "File size contract : PASS"
)


# ==============================================================================
# 11. STREAMED CONTENT AUDIT
# ==============================================================================

print("\n" + "=" * 90)
print("STREAMED CONTENT AUDIT")
print("=" * 90)

observed_rows = 0

seen_response_ids = set()
seen_response_objective_keys = set()

duplicate_response_ids = 0
duplicate_response_objective_keys = 0

empty_objective_rows = 0
empty_evidence_rows = 0

invalid_token_rows = 0

min_tokens = None
max_tokens = None

fold_counts = {}
target_counts = {}

required_stream_columns = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "objective_text",
    "evidence_text",
    "evidence_token_count",
    "target",
]

scanner = frozen_pf.iter_batches(
    batch_size=100_000
)

for batch_index, batch in enumerate(
    scanner,
    start=1,
):

    batch_df = (
        batch
        .to_pandas()
    )

    observed_rows += len(
        batch_df
    )

    # --------------------------------------------------------------------------
    # Required columns
    # --------------------------------------------------------------------------

    missing_batch_columns = sorted(
        set(required_stream_columns)
        -
        set(batch_df.columns)
    )

    assert not missing_batch_columns, (
        "Required streamed columns missing:\n"
        f"{missing_batch_columns}"
    )

    # --------------------------------------------------------------------------
    # Response identity
    # --------------------------------------------------------------------------

    response_values = (
        batch_df[
            "response_id"
        ]
        .astype(str)
    )

    objective_values = (
        batch_df[
            "objective_uid"
        ]
        .astype(str)
    )

    session_values = (
        batch_df[
            "session_id"
        ]
        .astype(str)
    )

    for response_id in response_values:

        if response_id in seen_response_ids:

            duplicate_response_ids += 1

        seen_response_ids.add(
            response_id
        )

    for response_id, objective_uid in zip(
        response_values,
        objective_values,
    ):

        key = (
            response_id,
            objective_uid,
        )

        if key in seen_response_objective_keys:

            duplicate_response_objective_keys += 1

        seen_response_objective_keys.add(
            key
        )

    # --------------------------------------------------------------------------
    # Text integrity
    # --------------------------------------------------------------------------

    objective_text = (
        batch_df[
            "objective_text"
        ]
    )

    evidence_text = (
        batch_df[
            "evidence_text"
        ]
    )

    empty_objective_rows += int(
        (
            objective_text.isna()
            |
            objective_text
            .astype(str)
            .str.strip()
            .eq("")
        ).sum()
    )

    empty_evidence_rows += int(
        (
            evidence_text.isna()
            |
            evidence_text
            .astype(str)
            .str.strip()
            .eq("")
        ).sum()
    )

    # --------------------------------------------------------------------------
    # Token integrity
    # --------------------------------------------------------------------------

    token_values = pd.to_numeric(
        batch_df[
            "evidence_token_count"
        ],
        errors="coerce",
    )

    token_array = (
        token_values
        .to_numpy(
            dtype=float
        )
    )

    invalid_mask = (
        ~np.isfinite(
            token_array
        )
        |
        (token_array < 1)
        |
        (token_array > 2048)
    )

    invalid_token_rows += int(
        invalid_mask.sum()
    )

    if len(token_values):

        batch_min = int(
            np.min(
                token_array
            )
        )

        batch_max = int(
            np.max(
                token_array
            )
        )

        if min_tokens is None:

            min_tokens = batch_min

        else:

            min_tokens = min(
                min_tokens,
                batch_min,
            )

        if max_tokens is None:

            max_tokens = batch_max

        else:

            max_tokens = max(
                max_tokens,
                batch_max,
            )

    # --------------------------------------------------------------------------
    # Fold distribution
    # --------------------------------------------------------------------------

    for value, count in (
        batch_df[
            "fold"
        ]
        .value_counts()
        .items()
    ):

        value = int(value)

        fold_counts[value] = (
            fold_counts.get(
                value,
                0,
            )
            +
            int(count)
        )

    # --------------------------------------------------------------------------
    # Target distribution
    # --------------------------------------------------------------------------

    for value, count in (
        batch_df[
            "target"
        ]
        .value_counts()
        .items()
    ):

        value = int(value)

        target_counts[value] = (
            target_counts.get(
                value,
                0,
            )
            +
            int(count)
        )

    if (
        batch_index
        %
        5
        ==
        0
    ):

        print(
            "Processed rows:",
            f"{observed_rows:,}",
        )

    del batch_df
    del token_values
    del token_array
    del invalid_mask

    gc.collect()


del scanner

gc.collect()


# ==============================================================================
# 12. EXACT CONTENT ASSERTIONS
# ==============================================================================

assert (
    observed_rows
    ==
    frozen_rows
), (
    "Streamed row count differs from parquet metadata."
)

assert (
    observed_rows
    ==
    EXPECTED_PACK_ROWS
), (
    "Streamed row population mismatch."
)

assert (
    duplicate_response_ids
    ==
    0
), (
    "Duplicate response_id detected."
)

assert (
    duplicate_response_objective_keys
    ==
    0
), (
    "Duplicate response-objective identity detected."
)

assert (
    empty_objective_rows
    ==
    0
), (
    "Empty objective_text rows detected."
)

assert (
    empty_evidence_rows
    ==
    0
), (
    "Empty evidence_text rows detected."
)

assert (
    invalid_token_rows
    ==
    0
), (
    "Invalid evidence_token_count rows detected."
)

assert (
    min_tokens
    is not None
), (
    "Minimum token count could not be determined."
)

assert (
    max_tokens
    is not None
), (
    "Maximum token count could not be determined."
)

assert (
    max_tokens
    <=
    2048
), (
    "Evidence token budget exceeded."
)

assert (
    set(
        fold_counts.keys()
    )
    ==
    {0, 1, 2, 3, 4}
), (
    "Frozen evidence does not contain exactly folds 0-4."
)

manifest_fold_counts = {
    int(key): int(value)
    for key, value in (
        R7_MANIFEST.get(
            "fold_counts",
            {}
        ).items()
    )
}

assert (
    fold_counts
    ==
    manifest_fold_counts
), (
    "Observed fold distribution differs from freeze manifest."
)

manifest_target_counts = {
    int(key): int(value)
    for key, value in (
        R7_MANIFEST.get(
            "target_counts",
            {}
        ).items()
    )
}

assert (
    target_counts
    ==
    manifest_target_counts
), (
    "Observed target distribution differs from freeze manifest."
)


print(
    "Streamed population : PASS"
)

print(
    "Identity uniqueness : PASS"
)

print(
    "Text integrity      : PASS"
)

print(
    "Token integrity     : PASS"
)

print(
    "Token range         :",
    f"{min_tokens} - {max_tokens}",
)

print(
    "Fold distribution   : PASS"
)

print(
    "Target distribution : PASS"
)


# ==============================================================================
# 13. MANIFEST TOKEN CONTRACT
# ==============================================================================

manifest_max_tokens = int(
    R7_MANIFEST.get(
        "max_evidence_tokens",
        -1,
    )
)

manifest_min_observed = int(
    R7_MANIFEST.get(
        "observed_min_evidence_tokens",
        -1,
    )
)

manifest_max_observed = int(
    R7_MANIFEST.get(
        "observed_max_evidence_tokens",
        -1,
    )
)

assert (
    manifest_max_tokens
    ==
    2048
), (
    "Manifest evidence token budget is not 2048."
)

assert (
    min_tokens
    ==
    manifest_min_observed
), (
    "Observed minimum token count differs from manifest."
)

assert (
    max_tokens
    ==
    manifest_max_observed
), (
    "Observed maximum token count differs from manifest."
)

print(
    "Manifest token contract : PASS"
)


# ==============================================================================
# 14. SAMPLE CONTENT RELOAD
#
# Read a deterministic small sample only.
# This is deliberately NOT the complete parquet.
# ==============================================================================

sample_table = frozen_pf.read_row_group(
    0,
    columns=[
        "response_id",
        "session_id",
        "objective_uid",
        "fold",
        "objective_text",
        "evidence_text",
        "evidence_token_count",
        "target",
    ],
)

sample_df = (
    sample_table
    .to_pandas()
    .head(10)
)

assert (
    len(sample_df)
    >
    0
), (
    "Frozen parquet first row group is empty."
)

assert (
    sample_df[
        "evidence_text"
    ]
    .astype(str)
    .str.strip()
    .ne("")
    .all()
), (
    "Sample contains empty evidence text."
)

assert (
    pd.to_numeric(
        sample_df[
            "evidence_token_count"
        ],
        errors="coerce",
    )
    .between(
        1,
        2048,
    )
    .all()
), (
    "Sample contains invalid token counts."
)

print("\n" + "=" * 90)
print("SAMPLE CONTENT RELOAD")
print("=" * 90)

print(
    "Rows inspected:",
    len(sample_df),
)

print(
    "Sample content : PASS"
)


# ==============================================================================
# 15. FINAL CELL 7 CONTRACT
# ==============================================================================

assert (
    FROZEN_EVIDENCE_PATH.exists()
), (
    "Frozen evidence artifact disappeared during audit."
)

assert (
    FREEZE_MANIFEST_PATH.exists()
), (
    "Freeze manifest disappeared during audit."
)

assert (
    sha256_observed
    ==
    sha256_expected
), (
    "Final SHA-256 integrity contract failed."
)

assert (
    schema_sha256_observed
    ==
    schema_sha256_expected
), (
    "Final schema integrity contract failed."
)

assert (
    frozen_rows
    ==
    EXPECTED_PACK_ROWS
), (
    "Final population contract failed."
)

assert (
    max_tokens
    <=
    2048
), (
    "Final evidence token contract failed."
)


# ==============================================================================
# 16. READY FLAG
# ==============================================================================

EVIDENCE_PACK_CELL_7_READY = True


print("\n" + "=" * 90)
print(
    "EVIDENCE PACK CELL 7 — "
    "FROZEN RELOAD VERIFICATION: PASS"
)
print("=" * 90)

print(
    "Frozen artifact:",
    FROZEN_EVIDENCE_PATH,
)

print(
    "Manifest:",
    FREEZE_MANIFEST_PATH,
)

print(
    "Rows:",
    f"{frozen_rows:,}",
)

print(
    "SHA-256:",
    sha256_observed,
)

print(
    "Token range:",
    f"{min_tokens} - {max_tokens}",
)

print(
    "Fold counts:",
    fold_counts,
)

print(
    "Target counts:",
    target_counts,
)


# ==============================================================================
# 17. MEMORY CLEANUP
# ==============================================================================

for _name in [
    "frozen_pf",
    "frozen_schema",
    "sample_table",
    "sample_df",
    "R7_MANIFEST",
]:
    if _name in globals():
        del globals()[_name]

gc.collect()

print(
    "Cell 7 memory cleanup: PASS"
)

TRACE THE ACE — EVIDENCE PACK BUILDER
CELL 7 — FROZEN RELOAD + EXACT INTEGRITY VERIFICATION

DEPENDENCY GATE
Cell 6 dependency : PASS

FROZEN ARTIFACT PATHS
Frozen root : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\03_evidence_pack\frozen
Candidates  : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\03_evidence_pack\frozen\evidence_packs.parquet
Manifest    : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\03_evidence_pack\frozen\cell6_freeze_manifest.json

ARTIFACT EXISTENCE
Frozen parquet : PASS
Freeze manifest: PASS

FREEZE MANIFEST
JSON valid : PASS
Status: FROZEN
Artifact: evidence_packs

FROZEN PARQUET METADATA
Rows: 35,072
Row groups: 1
Columns: ['response_id', 'session_id', 'objective_uid', 'fold', 'objective_text', 'evidence_text', 'evidence_token_count', 'selected_sections', 'source_turn_uids', 'source_turn_indices', 'source_roles', 'source_evidence_types', 'source_turn_count', 'source_min_turn_index', 'source_max_turn_index', 'has_stud

In [20]:
# ==============================================================================
# TRACE THE ACE — EVIDENCE PACK BUILDER
# CELL 8 — FINAL AUDIT / FREEZE GATE
# ==============================================================================

import gc
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


print("=" * 90)
print("TRACE THE ACE — EVIDENCE PACK BUILDER")
print("CELL 8 — FINAL AUDIT / FREEZE GATE")
print("=" * 90)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    "EVIDENCE_PACK_CELL_7_READY" in globals()
    and EVIDENCE_PACK_CELL_7_READY is True
), (
    "Cell 7 dependency failed."
)

print("\n" + "=" * 90)
print("DEPENDENCY GATE")
print("=" * 90)

print(
    "Cell 0 dependency : PASS"
)

print(
    "Cell 1 dependency : PASS"
)

print(
    "Cell 2 dependency : PASS"
)

print(
    "Cell 3 dependency : PASS"
)

print(
    "Cell 4 dependency : PASS"
)

print(
    "Cell 5 dependency : PASS"
)

print(
    "Cell 6 dependency : PASS"
)

print(
    "Cell 7 dependency : PASS"
)


# ==============================================================================
# 2. FROZEN PATHS
# ==============================================================================

PROJECT_ROOT = Path(
    PROJECT_ROOT
)

SCRATCH_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
)

EVIDENCE_ROOT = (
    SCRATCH_ROOT
    / "03_evidence_pack"
)

EVIDENCE_FREEZE_ROOT = (
    EVIDENCE_ROOT
    / "frozen"
)

FROZEN_EVIDENCE_PATH = (
    EVIDENCE_FREEZE_ROOT
    / "evidence_packs.parquet"
)

FREEZE_MANIFEST_PATH = (
    EVIDENCE_FREEZE_ROOT
    / "cell6_freeze_manifest.json"
)


print("\n" + "=" * 90)
print("FROZEN ARTIFACT PATHS")
print("=" * 90)

print(
    "Frozen root :",
    EVIDENCE_FREEZE_ROOT,
)

print(
    "Candidates  :",
    FROZEN_EVIDENCE_PATH,
)

print(
    "Manifest    :",
    FREEZE_MANIFEST_PATH,
)


# ==============================================================================
# 3. ARTIFACT EXISTENCE
# ==============================================================================

assert (
    EVIDENCE_FREEZE_ROOT.exists()
), (
    "Frozen evidence root is missing."
)

assert (
    FROZEN_EVIDENCE_PATH.exists()
), (
    "Frozen evidence parquet is missing."
)

assert (
    FREEZE_MANIFEST_PATH.exists()
), (
    "Frozen evidence manifest is missing."
)

print("\n" + "=" * 90)
print("FROZEN ARTIFACT EXISTENCE")
print("=" * 90)

print(
    "Evidence pack parquet : PASS"
)

print(
    "Freeze manifest       : PASS"
)


# ==============================================================================
# 4. LOAD MANIFEST
# ==============================================================================

with open(
    FREEZE_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as handle:

    manifest = json.load(
        handle
    )

assert (
    isinstance(
        manifest,
        dict,
    )
), (
    "Freeze manifest is not a JSON object."
)

assert (
    manifest.get(
        "status"
    )
    ==
    "FROZEN"
), (
    "Freeze manifest status is not FROZEN."
)

assert (
    manifest.get(
        "artifact"
    )
    ==
    "evidence_packs"
), (
    "Unexpected frozen artifact."
)

print("\n" + "=" * 90)
print("FREEZE MANIFEST")
print("=" * 90)

print(
    "JSON valid : PASS"
)

print(
    "Status:",
    manifest.get(
        "status"
    ),
)

print(
    "Artifact:",
    manifest.get(
        "artifact"
    ),
)


# ==============================================================================
# 5. PARQUET METADATA
# ==============================================================================

pf = pq.ParquetFile(
    FROZEN_EVIDENCE_PATH
)

schema = (
    pf.schema_arrow
)

columns = (
    schema.names
)

rows = (
    pf.metadata.num_rows
)

row_groups = (
    pf.metadata.num_row_groups
)


EXPECTED_ROWS = 35_072

assert (
    rows
    ==
    EXPECTED_ROWS
), (
    "Final evidence pack row count mismatch."
)

assert (
    rows
    ==
    int(
        manifest.get(
            "rows",
            -1,
        )
    )
), (
    "Manifest row count mismatch."
)


print("\n" + "=" * 90)
print("POPULATION CONTRACT")
print("=" * 90)

print(
    "Observed rows:",
    f"{rows:,}",
)

print(
    "Expected rows:",
    f"{EXPECTED_ROWS:,}",
)

print(
    "Population contract : PASS"
)


# ==============================================================================
# 6. FINAL SCHEMA CONTRACT
# ==============================================================================

REQUIRED_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "objective_text",
    "evidence_text",
    "evidence_token_count",
    "selected_sections",
    "source_turn_uids",
    "source_turn_indices",
    "source_roles",
    "source_evidence_types",
    "source_turn_count",
    "source_min_turn_index",
    "source_max_turn_index",
    "has_student_evidence",
    "has_tutor_context",
    "has_final_student_evidence",
    "target",
]

missing_columns = sorted(
    set(REQUIRED_COLUMNS)
    -
    set(columns)
)

assert not missing_columns, (
    "Frozen evidence is missing required columns:\n"
    f"{missing_columns}"
)

assert (
    columns
    ==
    manifest.get(
        "columns"
    )
), (
    "Frozen schema differs from manifest."
)


schema_sha256 = hashlib.sha256(
    str(
        schema
    ).encode(
        "utf-8"
    )
).hexdigest()

assert (
    schema_sha256
    ==
    manifest.get(
        "schema_sha256"
    )
), (
    "Final schema fingerprint mismatch."
)


print("\n" + "=" * 90)
print("SCHEMA CONTRACT")
print("=" * 90)

print(
    "Required columns : PASS"
)

print(
    "Schema fingerprint : PASS"
)


# ==============================================================================
# 7. FINAL SHA-256
# ==============================================================================

def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


final_sha256 = sha256_file(
    FROZEN_EVIDENCE_PATH
)

assert (
    final_sha256
    ==
    manifest.get(
        "sha256"
    )
), (
    "Final SHA-256 does not match freeze manifest."
)

assert (
    final_sha256
    ==
    manifest.get(
        "candidate_sha256"
    )
), (
    "Final candidate SHA-256 does not match manifest."
)


print("\n" + "=" * 90)
print("ARTIFACT HASH")
print("=" * 90)

print(
    "SHA-256:",
    final_sha256,
)

print(
    "SHA-256 integrity : PASS"
)


# ==============================================================================
# 8. FILE SIZE CONTRACT
# ==============================================================================

file_size = (
    FROZEN_EVIDENCE_PATH.stat()
    .st_size
)

assert (
    file_size
    ==
    int(
        manifest.get(
            "size_bytes",
            -1,
        )
    )
), (
    "Frozen file size differs from manifest."
)

assert (
    file_size
    >
    0
), (
    "Frozen evidence parquet is empty."
)

print(
    "File size:",
    f"{file_size:,} bytes",
)

print(
    "File size contract : PASS"
)


# ==============================================================================
# 9. FULL STREAMED FINAL AUDIT
# ==============================================================================

print("\n" + "=" * 90)
print("FULL STREAMED FINAL AUDIT")
print("=" * 90)

streamed_rows = 0

response_ids = set()
response_objective_keys = set()

duplicate_response_ids = 0
duplicate_response_objective_keys = 0

empty_objective = 0
empty_evidence = 0

invalid_tokens = 0

min_token_count = None
max_token_count = None

fold_counts = {}
target_counts = {}

scanner = pf.iter_batches(
    batch_size=100_000
)

for batch_index, batch in enumerate(
    scanner,
    start=1,
):

    df = (
        batch
        .to_pandas()
    )

    streamed_rows += len(
        df
    )

    # --------------------------------------------------------------------------
    # Identity
    # --------------------------------------------------------------------------

    response_values = (
        df[
            "response_id"
        ]
        .astype(str)
    )

    objective_values = (
        df[
            "objective_uid"
        ]
        .astype(str)
    )

    for response_id in response_values:

        if response_id in response_ids:

            duplicate_response_ids += 1

        response_ids.add(
            response_id
        )

    for response_id, objective_uid in zip(
        response_values,
        objective_values,
    ):

        key = (
            response_id,
            objective_uid,
        )

        if key in response_objective_keys:

            duplicate_response_objective_keys += 1

        response_objective_keys.add(
            key
        )

    # --------------------------------------------------------------------------
    # Text
    # --------------------------------------------------------------------------

    empty_objective += int(
        (
            df[
                "objective_text"
            ]
            .isna()
            |
            df[
                "objective_text"
            ]
            .astype(str)
            .str.strip()
            .eq("")
        ).sum()
    )

    empty_evidence += int(
        (
            df[
                "evidence_text"
            ]
            .isna()
            |
            df[
                "evidence_text"
            ]
            .astype(str)
            .str.strip()
            .eq("")
        ).sum()
    )

    # --------------------------------------------------------------------------
    # Tokens
    # --------------------------------------------------------------------------

    token_values = pd.to_numeric(
        df[
            "evidence_token_count"
        ],
        errors="coerce",
    )

    token_array = (
        token_values
        .to_numpy(
            dtype=float
        )
    )

    invalid_tokens += int(
        (
            ~np.isfinite(
                token_array
            )
            |
            (token_array < 1)
            |
            (token_array > 2048)
        ).sum()
    )

    if len(token_array):

        batch_min = int(
            np.min(
                token_array
            )
        )

        batch_max = int(
            np.max(
                token_array
            )
        )

        if min_token_count is None:

            min_token_count = (
                batch_min
            )

        else:

            min_token_count = min(
                min_token_count,
                batch_min,
            )

        if max_token_count is None:

            max_token_count = (
                batch_max
            )

        else:

            max_token_count = max(
                max_token_count,
                batch_max,
            )

    # --------------------------------------------------------------------------
    # Fold
    # --------------------------------------------------------------------------

    for value, count in (
        df[
            "fold"
        ]
        .value_counts()
        .items()
    ):

        value = int(value)

        fold_counts[value] = (
            fold_counts.get(
                value,
                0,
            )
            +
            int(count)
        )

    # --------------------------------------------------------------------------
    # Target
    # --------------------------------------------------------------------------

    for value, count in (
        df[
            "target"
        ]
        .value_counts()
        .items()
    ):

        value = int(value)

        target_counts[value] = (
            target_counts.get(
                value,
                0,
            )
            +
            int(count)
        )

    if (
        batch_index
        %
        5
        ==
        0
    ):

        print(
            "Processed:",
            f"{streamed_rows:,}",
            "rows",
        )

    del df
    del token_values
    del token_array

    gc.collect()


del scanner

gc.collect()


# ==============================================================================
# 10. FINAL STREAM ASSERTIONS
# ==============================================================================

assert (
    streamed_rows
    ==
    EXPECTED_ROWS
), (
    "Streamed row count mismatch."
)

assert (
    len(
        response_ids
    )
    ==
    EXPECTED_ROWS
), (
    "Unique response population mismatch."
)

assert (
    len(
        response_objective_keys
    )
    ==
    EXPECTED_ROWS
), (
    "Unique response-objective population mismatch."
)

assert (
    duplicate_response_ids
    ==
    0
), (
    "Duplicate response_id detected."
)

assert (
    duplicate_response_objective_keys
    ==
    0
), (
    "Duplicate response-objective identity detected."
)

assert (
    empty_objective
    ==
    0
), (
    "Empty objective text detected."
)

assert (
    empty_evidence
    ==
    0
), (
    "Empty evidence text detected."
)

assert (
    invalid_tokens
    ==
    0
), (
    "Invalid evidence token count detected."
)

assert (
    min_token_count
    ==
    int(
        manifest.get(
            "observed_min_evidence_tokens",
            -1,
        )
    )
), (
    "Minimum observed token count differs from manifest."
)

assert (
    max_token_count
    ==
    int(
        manifest.get(
            "observed_max_evidence_tokens",
            -1,
        )
    )
), (
    "Maximum observed token count differs from manifest."
)

assert (
    max_token_count
    <=
    2048
), (
    "Final evidence pack exceeds 2048 tokens."
)

assert (
    set(
        fold_counts.keys()
    )
    ==
    {0, 1, 2, 3, 4}
), (
    "Final fold coverage is incomplete."
)

manifest_fold_counts = {
    int(key): int(value)
    for key, value in (
        manifest.get(
            "fold_counts",
            {}
        ).items()
    )
}

assert (
    fold_counts
    ==
    manifest_fold_counts
), (
    "Final fold distribution differs from manifest."
)

manifest_target_counts = {
    int(key): int(value)
    for key, value in (
        manifest.get(
            "target_counts",
            {}
        ).items()
    )
}

assert (
    target_counts
    ==
    manifest_target_counts
), (
    "Final target distribution differs from manifest."
)


print(
    "Streamed population : PASS"
)

print(
    "Identity uniqueness : PASS"
)

print(
    "Text integrity      : PASS"
)

print(
    "Token integrity     : PASS"
)

print(
    "Fold coverage       : PASS"
)

print(
    "Target distribution : PASS"
)


# ==============================================================================
# 11. FINAL FROZEN CONTENT SUMMARY
# ==============================================================================

print("\n" + "=" * 90)
print("FINAL FROZEN CONTENT SUMMARY")
print("=" * 90)

print(
    "Responses:",
    f"{len(response_ids):,}",
)

print(
    "Response-objective packs:",
    f"{len(response_objective_keys):,}",
)

print(
    "Token range:",
    f"{min_token_count} - {max_token_count}",
)

print(
    "Fold counts:",
    fold_counts,
)

print(
    "Target counts:",
    target_counts,
)


# ==============================================================================
# 12. FINAL MANIFEST INTEGRITY CONTRACT
# ==============================================================================

manifest_integrity = (
    manifest.get(
        "integrity",
        {}
    )
)

assert (
    manifest_integrity.get(
        "population_contract"
    )
    is True
), (
    "Manifest population integrity flag is not true."
)

assert (
    manifest_integrity.get(
        "schema_contract"
    )
    is True
), (
    "Manifest schema integrity flag is not true."
)

assert (
    manifest_integrity.get(
        "duplicate_response_id_check"
    )
    is True
), (
    "Manifest response identity flag is not true."
)

assert (
    manifest_integrity.get(
        "duplicate_response_objective_check"
    )
    is True
), (
    "Manifest response-objective identity flag is not true."
)

assert (
    manifest_integrity.get(
        "text_integrity"
    )
    is True
), (
    "Manifest text integrity flag is not true."
)

assert (
    manifest_integrity.get(
        "token_budget_contract"
    )
    is True
), (
    "Manifest token budget flag is not true."
)

assert (
    manifest_integrity.get(
        "fold_coverage"
    )
    is True
), (
    "Manifest fold coverage flag is not true."
)

assert (
    manifest_integrity.get(
        "sha256_verified"
    )
    is True
), (
    "Manifest SHA-256 flag is not true."
)

print(
    "Manifest integrity flags : PASS"
)


# ==============================================================================
# 13. FINAL FREEZE GATE
# ==============================================================================

EVIDENCE_PACK_CELL_8_READY = True

print("\n" + "=" * 90)
print(
    "EVIDENCE PACK CELL 8 — "
    "FINAL AUDIT / FREEZE GATE: PASS"
)
print("=" * 90)

print(
    "Frozen evidence:",
    FROZEN_EVIDENCE_PATH,
)

print(
    "Freeze manifest:",
    FREEZE_MANIFEST_PATH,
)

print(
    "Rows:",
    f"{streamed_rows:,}",
)

print(
    "SHA-256:",
    final_sha256,
)

print(
    "Token range:",
    f"{min_token_count} - {max_token_count}",
)

print(
    "Fold counts:",
    fold_counts,
)

print(
    "Target counts:",
    target_counts,
)


# ==============================================================================
# 14. MEMORY CLEANUP
# ==============================================================================

for _name in [
    "pf",
    "schema",
    "manifest",
]:
    if _name in globals():
        del globals()[_name]

gc.collect()

print(
    "Cell 8 memory cleanup: PASS"
)

TRACE THE ACE — EVIDENCE PACK BUILDER
CELL 8 — FINAL AUDIT / FREEZE GATE

DEPENDENCY GATE
Cell 0 dependency : PASS
Cell 1 dependency : PASS
Cell 2 dependency : PASS
Cell 3 dependency : PASS
Cell 4 dependency : PASS
Cell 5 dependency : PASS
Cell 6 dependency : PASS
Cell 7 dependency : PASS

FROZEN ARTIFACT PATHS
Frozen root : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\03_evidence_pack\frozen
Candidates  : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\03_evidence_pack\frozen\evidence_packs.parquet
Manifest    : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\03_evidence_pack\frozen\cell6_freeze_manifest.json

FROZEN ARTIFACT EXISTENCE
Evidence pack parquet : PASS
Freeze manifest       : PASS

FREEZE MANIFEST
JSON valid : PASS
Status: FROZEN
Artifact: evidence_packs

POPULATION CONTRACT
Observed rows: 35,072
Expected rows: 35,072
Population contract : PASS

SCHEMA CONTRACT
Required columns : PASS
Schema fingerprint : PASS

ARTIFACT HASH
SHA-256: 